# Démonstration visuelle : vision par ordinateur, IA multimodale et métriques d'évaluation

Notebook compagnon de la présentation *Vision par ordinateur et IA multimodale : applications et métriques d'évaluation* (M. Haddassy, Master 2 Data & IA, INPT Rabat). Chaque section suit la même chaîne : **données → modèle → prédiction → visualisation → évaluation → interprétation**.

| Section | Tâche | Diapositives | Modèle exécuté | Métriques |
|---|---|---|---|---|
| 0 | Entraîner, valider, tester | 3 | petit réseau **entraîné ici** | perte, exactitude |
| 1 | Suivi d'objets | 4 – 6 | YOLO11n + ByteTrack | MOTA, IDF1, HOTA |
| 2 | Génération d'images | 7 – 9 | Stable Diffusion (GPU) · CLIP | FID, CLIPScore |
| 3 | OCR | 10 – 12 | Tesseract 5 | CER, WER, correspondance exacte |
| 4 | Légendage | 13 – 15 | BLIP | BLEU, METEOR, CIDEr |
| 5 | Recherche image-texte | 16 – 18 | CLIP | Recall@K, Precision@K, MRR |

Chaque figure porte une étiquette : **RÉSULTAT RÉEL** (modèle réellement exécuté), **CALCUL PÉDAGOGIQUE** (exemple construit à la main, souvent celui de la diapositive) ou **ILLUSTRATION CONCEPTUELLE** (schéma). **Mode A** : CPU, rapide. **Mode B** : GPU, avec génération d'images.

## Préparation (à exécuter une fois)

In [ ]:
# ===================== PARAMÈTRES DE LA DÉMONSTRATION =====================
MODE = "A"                     # "A" = démonstration rapide (CPU) ; "B" = démonstration complète (GPU)
GENERER_IMAGE_SUR_CPU = False  # True : tente la génération d'image sans GPU (SD-Turbo : ~5 Go à télécharger, plusieurs minutes)
INSTALLER_DEPENDANCES = None   # None = automatique (installe seulement dans Google Colab) ; True ou False pour forcer
GRAINE = 42                    # graine aléatoire : rend les résultats reproductibles

In [ ]:
import os, sys, subprocess

EN_COLAB = "google.colab" in sys.modules
if INSTALLER_DEPENDANCES is None:
    INSTALLER_DEPENDANCES = EN_COLAB


def pip_installer(*paquets):
    # Installation silencieuse ; une erreur n'interrompt pas le notebook (elle sera signalée plus loin).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paquets], check=False)


if INSTALLER_DEPENDANCES:
    pip_installer("ultralytics>=8.3", "lap>=0.5.12", "open_clip_torch>=2.24", "motmetrics>=1.4",
                  "jiwer>=3.0", "pytesseract>=0.3.10", "nltk>=3.8", "ipywidgets>=8")
    if MODE == "B":
        pip_installer("diffusers>=0.30", "accelerate>=0.30")
    if EN_COLAB:
        # moteur OCR (programme externe) + modèle de langue française
        subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr tesseract-ocr-fra > /dev/null",
                       shell=True, check=False)
    print("Installation terminée.")
else:
    print("Installation automatique ignorée (environnement local) : voir README.md si un module manque.")

In [ ]:
import zipfile
from pathlib import Path


def trouver_racine():
    # Cherche le dossier qui contient « assets/ » : dossier courant, sous-dossier du projet, ou emplacements Colab usuels.
    candidats = [Path.cwd(), Path.cwd() / "Projet_Demonstration_IA_Multimodale",
                 Path("/content/Projet_Demonstration_IA_Multimodale"),
                 Path("/content/drive/MyDrive/Projet_Demonstration_IA_Multimodale")]
    for c in candidats:
        if (c / "assets").is_dir() and (c / "assets" / "SOURCES.md").exists():
            return c.resolve()
    return None


RACINE = trouver_racine()
if RACINE is None and EN_COLAB:
    from google.colab import files
    print("Téléversez l'archive Projet_Demonstration_IA_Multimodale.zip :")
    for nom in files.upload():
        if nom.endswith(".zip"):
            zipfile.ZipFile(nom).extractall("/content")
    RACINE = trouver_racine()
if RACINE is None:
    raise FileNotFoundError("Dossier « assets/ » introuvable : ouvrez ce notebook depuis la racine du projet (voir README.md).")

os.chdir(RACINE)
ASSETS, MODELES, SORTIES = RACINE / "assets", RACINE / "modeles", RACINE / "resultats"
MODELES.mkdir(exist_ok=True)
SORTIES.mkdir(exist_ok=True)

FICHIERS_ATTENDUS = [
    "suivi/nuscenes_mini.mp4", "suivi/MOT17-04-FRCNN_extrait/gt/gt.txt", "suivi/MOT17-04-FRCNN_extrait/img1/000001.jpg",
    "generation/cafe_scikit_image.jpg", "ocr/facture_scan.jpg", "ocr/facture_scan_degradee.jpg", "ocr/facture_reference.json",
    "legendage/quai_metro_2414384480.jpg", "legendage/references.json", "recherche/collection.json",
]
manquants = [f for f in FICHIERS_ATTENDUS if not (ASSETS / f).exists()]
print(f"Dossier du projet : {RACINE}")
print("Ressources : toutes présentes." if not manquants else f"Ressources manquantes : {manquants}")

In [ ]:
import json, math, platform, random, re, textwrap, time, unicodedata, warnings
from collections import Counter
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import patches
from PIL import Image
from IPython.display import HTML, Markdown, display

import torch

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Graines aléatoires : mêmes résultats d'une exécution à l'autre (à matériel égal)
random.seed(GRAINE)
np.random.seed(GRAINE)
torch.manual_seed(GRAINE)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
GPU = DEVICE == "cuda"
DEVICE_YOLO = 0 if GPU else DEVICE  # syntaxe attendue par Ultralytics

print(f"Python {platform.python_version()} · PyTorch {torch.__version__} · matériel : {DEVICE.upper()} · mode {MODE}")
if GPU:
    proprietes = torch.cuda.get_device_properties(0)
    print(f"GPU : {proprietes.name} · mémoire : {proprietes.total_memory / 1024**3:.1f} Go")
if MODE == "B" and not GPU:
    print("Attention : mode B demandé, mais aucun GPU CUDA n'est disponible. "
          "La génération d'image sera ignorée, sauf si GENERER_IMAGE_SUR_CPU = True.")

In [ ]:
PALETTE = {"fond": "#F7F8FA", "surface": "#EDF2F5", "marine": "#19334A", "sarcelle": "#287B83",
           "sauge": "#72978C", "terracotta": "#C17B64", "gris": "#8A97A3", "ligne": "#DCE4E8", "ocre": "#B8964A"}
COULEURS_PISTES = ["#287B83", "#C17B64", "#4E6E9E", "#B8964A", "#72978C", "#8E5B8C", "#19334A",
                   "#D08C3E", "#5B8FA8", "#A35D5D", "#6E8B3D", "#7A6A9B"]

plt.rcParams.update({
    "figure.facecolor": PALETTE["fond"], "axes.facecolor": PALETTE["fond"], "savefig.facecolor": PALETTE["fond"],
    "text.color": PALETTE["marine"], "axes.labelcolor": PALETTE["marine"], "axes.edgecolor": PALETTE["gris"],
    "xtick.color": PALETTE["marine"], "ytick.color": PALETTE["marine"], "axes.titlecolor": PALETTE["marine"],
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12, "figure.titlesize": 18,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False, "figure.dpi": 100,
})

ETIQUETTES = {"reel": ("RÉSULTAT RÉEL", PALETTE["sarcelle"]),
              "pedagogique": ("CALCUL PÉDAGOGIQUE", PALETTE["ocre"]),
              "conceptuel": ("ILLUSTRATION CONCEPTUELLE", PALETTE["gris"]),
              "donnees": ("DONNÉES D'ENTRÉE", PALETTE["marine"])}


def fmt(x, decimales=2, pct=False):
    # Écriture française des nombres : 0,70 ; pct=True → 70,0 % ; valeur manquante → « — »
    if x is None or (isinstance(x, (float, np.floating)) and np.isnan(x)):
        return "—"
    if pct:
        return f"{100 * x:.1f}".replace(".", ",") + " %"
    return f"{x:.{decimales}f}".replace(".", ",")


def etiqueter(fig, type_resultat, detail=""):
    # Ajoute dans le coin supérieur droit le type de résultat (réel, pédagogique ou conceptuel).
    texte, couleur = ETIQUETTES[type_resultat]
    fig.text(0.995, 0.995, texte + (f"  ·  {detail}" if detail else ""), ha="right", va="top", fontsize=10.5,
             color="white", fontweight="bold", bbox=dict(boxstyle="round,pad=0.35", fc=couleur, ec="none"))


def interpretation(texte):
    # Courte interprétation affichée sous une figure ou un tableau.
    display(Markdown(f"> **Lecture.** {texte}"))


def sauvegarder(fig, nom):
    fig.savefig(SORTIES / nom, dpi=130, bbox_inches="tight")


@contextmanager
def chrono(libelle):
    debut = time.perf_counter()
    yield
    print(f"{libelle} : {time.perf_counter() - debut:.1f} s")


def schema_pipeline(etapes, titre, sous_titres=None, mise_en_avant=None):
    # Dessine une chaîne de traitement « étape → étape → … » (illustration conceptuelle réutilisée dans chaque section).
    n = len(etapes)
    fig, ax = plt.subplots(figsize=(15, 2.7))
    ax.set_xlim(0, n)
    ax.set_ylim(0, 1)
    ax.set_axis_off()
    for i, nom in enumerate(etapes):
        couleur = PALETTE["sarcelle"] if mise_en_avant == i else "white"
        ax.add_patch(patches.FancyBboxPatch((i + 0.03, 0.22), 0.86, 0.56, boxstyle="round,pad=0.01,rounding_size=0.03",
                                            fc=couleur, ec=PALETTE["sarcelle"], lw=1.6))
        ax.text(i + 0.46, 0.58, nom, ha="center", va="center", fontsize=13.5, fontweight="bold",
                color="white" if mise_en_avant == i else PALETTE["marine"])
        if sous_titres:
            ax.text(i + 0.46, 0.36, sous_titres[i], ha="center", va="center", fontsize=11,
                    color="white" if mise_en_avant == i else PALETTE["gris"])
        if i < n - 1:
            ax.annotate("", xy=(i + 1.025, 0.5), xytext=(i + 0.9, 0.5),
                        arrowprops=dict(arrowstyle="-|>", color=PALETTE["marine"], lw=1.6))
    fig.suptitle(titre, x=0.01, ha="left", fontsize=15, fontweight="bold")
    etiqueter(fig, "conceptuel")
    plt.show()


def vignette(image, ratio=4 / 3):
    # Image entière placée dans un cadre 4:3 (bandes de la couleur du fond) : vignettes de même taille, sans rien couper
    l, h = image.size
    L, H = (max(l, int(h * ratio)), h) if l / h < ratio else (l, max(h, int(l / ratio)))
    cadre = Image.new("RGB", (L, H), PALETTE["fond"])
    cadre.paste(image.convert("RGB"), ((L - l) // 2, (H - h) // 2))
    return cadre


try:  # contrôles interactifs (facultatifs) : sliders, menus, zones de texte
    import ipywidgets as widgets
    WIDGETS_OK = True
except ImportError:
    WIDGETS_OK = False


def explorer(fonction, **controles):
    # Vue initiale (affichage normal, visible aussi dans les exports), puis contrôles ipywidgets :
    # chaque changement relance « fonction » et affiche le nouveau résultat sous les contrôles.
    fonction(**{nom: w.value for nom, w in controles.items()})
    plt.show()
    sortie = widgets.Output()

    def mettre_a_jour(*_):
        with sortie:
            sortie.clear_output(wait=True)
            fonction(**{nom: w.value for nom, w in controles.items()})
            plt.show()

    for w in controles.values():
        w.observe(mettre_a_jour, names="value")
    display(widgets.VBox([widgets.HTML("<b>Exploration interactive</b> : modifiez un contrôle, le résultat s'affiche ci-dessous."),
                          widgets.HBox(list(controles.values())), sortie]))


RESULTATS = {}  # résultats mesurés de chaque section, réutilisés dans la synthèse finale
print("Configuration graphique prête.")

In [ ]:
URL_CLIP_REPLI = ("https://github.com/mlfoundations/open_clip/releases/download/v0.2-weights/"
                  "vit_b_32-quickgelu-laion400m_e32-46683a32.pt")


def charger_clip():
    import open_clip  # importé ici : les sections qui n'utilisent pas CLIP restent exécutables sans ce paquet

    # 1) Modèle de référence de CLIPScore : CLIP ViT-B/32 d'OpenAI
    try:
        modele, _, pretraitement = open_clip.create_model_and_transforms("ViT-B-32", pretrained="openai", device=DEVICE)
        return {"modele": modele.eval(), "pretraitement": pretraitement, "tokeniseur": open_clip.get_tokenizer("ViT-B-32"),
                "nom": "CLIP ViT-B/32 (OpenAI)", "clipscore_strict": True}
    except Exception as erreur:
        print(f"CLIP d'OpenAI indisponible ({type(erreur).__name__}) : chargement du modèle de repli.")
    # 2) Repli : même architecture, entraînée sur LAION-400M (fichier hébergé sur GitHub)
    chemin = MODELES / URL_CLIP_REPLI.rsplit("/", 1)[-1]
    if not chemin.exists():
        torch.hub.download_url_to_file(URL_CLIP_REPLI, str(chemin))
    modele, _, pretraitement = open_clip.create_model_and_transforms("ViT-B-32-quickgelu", pretrained=str(chemin), device=DEVICE)
    return {"modele": modele.eval(), "pretraitement": pretraitement, "tokeniseur": open_clip.get_tokenizer("ViT-B-32-quickgelu"),
            "nom": "OpenCLIP ViT-B/32 (LAION-400M, repli)", "clipscore_strict": False}


def encoder_images(clip, images):
    with torch.no_grad():
        x = torch.stack([clip["pretraitement"](im.convert("RGB")) for im in images]).to(DEVICE)
        return torch.nn.functional.normalize(clip["modele"].encode_image(x).float(), dim=-1).cpu()


def encoder_textes(clip, textes):
    with torch.no_grad():
        jetons = clip["tokeniseur"](textes).to(DEVICE)
        return torch.nn.functional.normalize(clip["modele"].encode_text(jetons).float(), dim=-1).cpu()


print("Fonctions CLIP prêtes (modèle chargé à la demande).")

---
# 0. Comment un modèle est-il développé et évalué ? — diapositive 3

On découpe des données **annotées** en trois parties : l'**entraînement** ajuste les poids, la **validation** sert à régler le modèle (et à choisir quand s'arrêter), le **test** mesure la performance finale sur des données jamais vues.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 3.4))
ax.set_xlim(0, 16)
ax.set_ylim(0, 3.4)
ax.set_axis_off()


def bloc(x, y, l, h, texte, fond, couleur_texte=PALETTE["marine"], taille=13.5):
    ax.add_patch(patches.FancyBboxPatch((x, y), l, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc=fond, ec=PALETTE["sarcelle"], lw=1.5))
    ax.text(x + l / 2, y + h / 2, texte, ha="center", va="center", fontsize=taille, fontweight="bold", color=couleur_texte)


def fleche(x1, y1, x2, y2, texte=""):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=PALETTE["marine"], lw=1.6))
    if texte:
        ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.14, texte, ha="center", va="bottom", fontsize=11.5, color=PALETTE["gris"])


bloc(0.1, 1.2, 2.2, 1.0, "Données\nannotées", "white")
bloc(3.2, 2.35, 2.6, 0.75, "Entraînement 60 %", PALETTE["surface"])
bloc(3.2, 1.33, 2.6, 0.75, "Validation 20 %", "#C9DCDF")
bloc(3.2, 0.31, 2.6, 0.75, "Test 20 %", PALETTE["sarcelle"], "white")
for y in (2.72, 1.7, 0.68):
    fleche(2.35, 1.7, 3.15, y)
bloc(8.6, 1.95, 2.2, 1.15, "Modèle", "white")
fleche(5.85, 2.72, 8.55, 2.72, "ajuster les poids")
fleche(5.85, 1.7, 8.55, 2.15, "régler, choisir l'époque")
bloc(10.9, 0.3, 2.4, 1.15, "Prédictions\nsur le test", "white")
fleche(9.7, 1.9, 11.4, 1.5)
fleche(5.85, 0.68, 10.85, 0.68, "données jamais vues")
bloc(14.0, 0.3, 1.9, 1.15, "Métrique", PALETTE["sarcelle"], "white")
fleche(13.35, 0.88, 13.95, 0.88)
ax.text(14.0 + 0.95, 1.7, "vérité terrain\ndu test", ha="center", fontsize=11.5, color=PALETTE["sarcelle"])
fig.suptitle("Données → entraînement / validation / test → modèle → évaluation", x=0.01, ha="left", fontweight="bold")
etiqueter(fig, "conceptuel")
plt.show()

display(Markdown("**Dans ce notebook, les cinq modèles sont déjà entraînés par leurs auteurs : on ne fait que de l'inférence, puis on évalue sur des exemples annotés.**"))
display(pd.DataFrame([
    ["1. Suivi", "YOLO11n + ByteTrack", "COCO (images annotées de 80 classes)", "extrait MOT17-04 : boîtes et identités officielles"],
    ["2. Génération", "Stable Diffusion · CLIP", "paires image-texte du web (LAION ; WIT-400M pour CLIP)", "aucune image de référence unique : distributions"],
    ["3. OCR", "Tesseract 5 (LSTM)", "lignes de texte rendues dans de nombreuses polices", "facture : transcription vérifiée"],
    ["4. Légendage", "BLIP", "paires image-texte (web, COCO…)", "photo Flickr8k : 5 légendes humaines"],
    ["5. Recherche", "CLIP", "400 millions de paires image-texte", "4 requêtes étiquetées à la main"],
], columns=["Section", "Modèle", "Entraîné par ses auteurs sur", "Évalué ici sur (vérité terrain)"]).set_index("Section"))

### Un vrai entraînement, en quelques secondes

Pour voir de **vraies** courbes d'entraînement et de validation, on entraîne ici un petit réseau convolutif à reconnaître des **chiffres manuscrits** (8 × 8 pixels, jeu `digits` de scikit-learn, inclus sans téléchargement) : c'est la brique de base d'un OCR.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

chiffres = load_digits()
X = chiffres.images.astype(np.float32) / 16.0  # pixels ramenés entre 0 et 1
y = chiffres.target
X_ent, X_tmp, y_ent, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=GRAINE)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=GRAINE)
print(f"Entraînement : {len(X_ent)} images · validation : {len(X_val)} · test : {len(X_test)}")

fig, axes = plt.subplots(3, 10, figsize=(15, 4.9))
for ligne, (nom, Xs, ys) in enumerate([("entraînement", X_ent, y_ent), ("validation", X_val, y_val), ("test", X_test, y_test)]):
    for k in range(10):
        ax = axes[ligne, k]
        ax.imshow(Xs[k], cmap="Greys")
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f"étiquette {ys[k]}", fontsize=10)
    axes[ligne, 0].set_ylabel(nom, fontsize=13, fontweight="bold")
fig.suptitle("Trois ensembles disjoints, chacun avec ses étiquettes (vérité terrain)", x=0.01, ha="left", fontweight="bold")
fig.subplots_adjust(top=0.84, hspace=0.45)
etiqueter(fig, "donnees", "scikit-learn digits")
plt.show()

In [ ]:
import torch.nn as nn

torch.manual_seed(GRAINE)
reseau = nn.Sequential(
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 8×8 → 4×4
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 4×4 → 2×2
    nn.Flatten(), nn.Linear(32 * 2 * 2, 64), nn.ReLU(), nn.Linear(64, 10))
perte_fn = nn.CrossEntropyLoss()  # entropie croisée : perte de classification
optimiseur = torch.optim.Adam(reseau.parameters(), lr=3e-3)
tenseurs = lambda Xs, ys: (torch.tensor(Xs).unsqueeze(1), torch.tensor(ys))
(Xe, ye), (Xv, yv), (Xt, yt) = tenseurs(X_ent, y_ent), tenseurs(X_val, y_val), tenseurs(X_test, y_test)


def evaluer(Xs, ys):
    reseau.eval()
    with torch.no_grad():
        sorties = reseau(Xs)
    return perte_fn(sorties, ys).item(), (sorties.argmax(1) == ys).float().mean().item()


historique, meilleur = [], (float("inf"), None, 0)
with chrono("Entraînement (40 époques, CPU)"):
    for epoque in range(1, 41):
        reseau.train()
        for lot in torch.randperm(len(Xe), generator=torch.Generator().manual_seed(epoque)).split(64):
            optimiseur.zero_grad()
            perte_fn(reseau(Xe[lot]), ye[lot]).backward()
            optimiseur.step()
        (pe, ae), (pv, av) = evaluer(Xe, ye), evaluer(Xv, yv)
        historique.append({"époque": epoque, "perte entraînement": pe, "perte validation": pv,
                           "exactitude entraînement": ae, "exactitude validation": av})
        if pv < meilleur[0]:  # on garde les poids de la meilleure époque sur la VALIDATION (arrêt précoce)
            meilleur = (pv, {k: v.clone() for k, v in reseau.state_dict().items()}, epoque)
historique = pd.DataFrame(historique).set_index("époque")
reseau.load_state_dict(meilleur[1])
perte_test, exactitude_test = evaluer(Xt, yt)  # le test n'est utilisé qu'une seule fois, à la fin

fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
for ax, (cle, titre) in zip(axes, [("perte", "Perte (entropie croisée) : plus bas = meilleur"), ("exactitude", "Exactitude : plus haut = meilleur")]):
    ax.plot(historique.index, historique[f"{cle} entraînement"], color=PALETTE["sarcelle"], lw=2.4, label="entraînement")
    ax.plot(historique.index, historique[f"{cle} validation"], color=PALETTE["terracotta"], lw=2.4, label="validation")
    ax.axvline(meilleur[2], color=PALETTE["gris"], ls=":", lw=1.5)
    ax.set_xlabel("Époque")
    ax.set_title(titre, loc="left", fontsize=13)
    ax.legend()
axes[0].text(meilleur[2] + 0.5, axes[0].get_ylim()[1] * 0.85, f"meilleure validation :\népoque {meilleur[2]}", fontsize=11, color=PALETTE["gris"])
fig.suptitle("Courbes réelles d'entraînement et de validation", x=0.01, ha="left", fontweight="bold")
fig.subplots_adjust(top=0.84)
etiqueter(fig, "reel", "entraînement exécuté dans ce notebook")
sauvegarder(fig, "entrainement_courbes.png")
plt.show()
RESULTATS["entrainement"] = {"exactitude_test": exactitude_test, "epoque": meilleur[2]}
interpretation(f"Exactitude finale sur le **test** : {fmt(exactitude_test, pct=True)} ({len(X_test)} images jamais vues). "
               f"La perte de validation est la plus basse à l'époque {meilleur[2]} : on garde ces poids. "
               "Quand la perte d'entraînement continue de baisser alors que celle de validation stagne ou remonte, le modèle commence à **sur-apprendre**.")

In [ ]:
reseau.eval()
with torch.no_grad():
    predictions_test = reseau(Xt).argmax(1).numpy()
matrice = confusion_matrix(y_test, predictions_test)
erreurs = np.where(predictions_test != y_test)[0]

fig = plt.figure(figsize=(16, 5.4))
grille = fig.add_gridspec(2, 7, width_ratios=[2.3, 2.3, 1, 1, 1, 1, 1], wspace=0.35)
ax = fig.add_subplot(grille[:, :2])
ax.imshow(matrice, cmap="BuGn")
for i in range(10):
    for j in range(10):
        if matrice[i, j]:
            ax.text(j, i, matrice[i, j], ha="center", va="center", fontsize=10,
                    color="white" if matrice[i, j] > matrice.max() / 2 else (PALETTE["terracotta"] if i != j else PALETTE["marine"]),
                    fontweight="bold" if i != j else "normal")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("Chiffre prédit")
ax.set_ylabel("Vérité terrain")
ax.set_title("Matrice de confusion (test)", loc="left", fontsize=13)
for k, i in enumerate(erreurs[:10]):
    a = fig.add_subplot(grille[k // 5, 2 + k % 5])
    a.imshow(X_test[i], cmap="Greys")
    a.set_xticks([])
    a.set_yticks([])
    a.set_title(f"vrai {y_test[i]} → lu {predictions_test[i]}", fontsize=10.5, color=PALETTE["terracotta"])
fig.suptitle(f"Erreurs du modèle sur le test : {len(erreurs)} sur {len(y_test)}", x=0.01, ha="left", fontweight="bold")
etiqueter(fig, "reel", "évaluation sur le test")
plt.show()
interpretation("La diagonale contient les bonnes réponses ; chaque case hors diagonale est une confusion précise"
               + (f" (par exemple un {y_test[erreurs[0]]} lu {predictions_test[erreurs[0]]}). " if len(erreurs) else ". ")
               +
               "C'est le même principe que l'OCR de la section 3 : comparer la prédiction à la vérité terrain, caractère par caractère.")

---
# 1. Suivi d'objets — diapositives 4 à 6

**Objectif.** Suivre les véhicules d'une vidéo de trafic en gardant le même identifiant, puis mesurer la qualité du suivi (MOTA, IDF1, HOTA) sur des annotations officielles.

**Modèle** : YOLO11n (détection, pré-entraîné sur COCO) + ByteTrack (association). **Vérité terrain** : extrait de MOT17-04 (section 1.3).

In [ ]:
schema_pipeline(["Images vidéo", "Détection", "Association", "Identités"],
                "Chaîne de suivi « tracking-by-detection »",
                ["suite temporelle", "YOLO11n : boîtes", "ByteTrack : t → t + 1", "trajectoires"], mise_en_avant=2)

### 1.1 Entrée : la vidéo de la diapositive 4
Une scène nuScenes complète (Boston) : 40 **images clés** prises toutes les 0,5 s, soit 20 s ; le fichier vidéo les rejoue à 20 images/s.

In [ ]:
import cv2

VIDEO = ASSETS / "suivi" / "nuscenes_mini.mp4"
capture = cv2.VideoCapture(str(VIDEO))
FPS_VIDEO = capture.get(cv2.CAP_PROP_FPS)  # cadence de lecture du fichier
PAS_TEMPS = 0.5  # écart réel entre deux images clés nuScenes (2 images par seconde)
images_video = []
while True:
    ok, image_bgr = capture.read()
    if not ok:
        break
    images_video.append(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))  # OpenCV lit en BGR, Matplotlib attend du RGB
capture.release()

H_VID, L_VID = images_video[0].shape[:2]
print(f"Vidéo : {len(images_video)} images clés de {L_VID} × {H_VID} px, une toutes les {fmt(PAS_TEMPS, 1)} s, "
      f"soit {fmt(len(images_video) * PAS_TEMPS, 0)} s de scène (fichier lu à {FPS_VIDEO:.0f} images/s).")

INDICES_AFFICHES = np.linspace(0, len(images_video) - 1, 4).round().astype(int)
fig, axes = plt.subplots(1, 4, figsize=(19, 3.6))
for ax, i in zip(axes, INDICES_AFFICHES):
    ax.imshow(images_video[i])
    ax.set_axis_off()
    ax.set_title(f"image {i} · t = {fmt(i * PAS_TEMPS, 1)} s", fontsize=13)
fig.suptitle("Entrée : quatre images de la séquence nuScenes", x=0.01, ha="left", fontweight="bold")
etiqueter(fig, "donnees")
plt.show()

### 1.2 Prédiction : détection + suivi

In [ ]:
from ultralytics import YOLO

CLASSES_VEHICULES = {2: "voiture", 3: "moto", 5: "bus", 7: "camion"}  # identifiants des classes COCO
modele_suivi = YOLO(str(MODELES / "yolo11n.pt"))

lignes = []
with chrono("Détection + suivi sur toute la vidéo"):
    flux = modele_suivi.track(source=str(VIDEO), stream=True, persist=True, tracker="bytetrack.yaml",
                              classes=list(CLASSES_VEHICULES), conf=0.25, device=DEVICE_YOLO, verbose=False)
    for i, res in enumerate(flux):
        if res.boxes.id is None:  # aucune piste confirmée dans cette image
            continue
        for tid, cls, conf, (x1, y1, x2, y2) in zip(res.boxes.id.int().tolist(), res.boxes.cls.int().tolist(),
                                                    res.boxes.conf.tolist(), res.boxes.xyxy.tolist()):
            lignes.append({"image": i, "id_brut": tid, "classe": CLASSES_VEHICULES.get(cls, str(cls)),
                           "confiance": conf, "x1": x1, "y1": y1, "x2": x2, "y2": y2})

pistes = pd.DataFrame(lignes, columns=["image", "id_brut", "classe", "confiance", "x1", "y1", "x2", "y2"])
# Renumérotation 1, 2, 3… par ordre d'apparition : les identifiants internes de ByteTrack ne sont pas consécutifs
ordre = pistes.groupby("id_brut")["image"].min().sort_values(kind="stable").index
pistes["id"] = pistes["id_brut"].map({brut: k + 1 for k, brut in enumerate(ordre)})
duree_pistes = pistes.groupby("id")["image"].agg(premiere="min", derniere="max", nb_images="count")

print(f"{len(pistes)} boîtes suivies · {pistes['id'].nunique()} identifiants · "
      f"piste la plus longue : {duree_pistes['nb_images'].max()} images sur {len(images_video)}")
display(pistes.drop(columns="id_brut").head(6).round(1))

In [ ]:
def couleur_piste(k):
    return COULEURS_PISTES[(int(k) - 1) % len(COULEURS_PISTES)]


def dessiner_suivi(ax, i, trajectoires=False):
    ax.imshow(images_video[i])
    ax.set_axis_off()
    ax.set_title(f"image {i} · t = {fmt(i * PAS_TEMPS, 1)} s", fontsize=14)
    for _, b in pistes[pistes.image == i].iterrows():
        c = couleur_piste(b.id)
        ax.add_patch(patches.Rectangle((b.x1, b.y1), b.x2 - b.x1, b.y2 - b.y1, fill=False, ec=c, lw=2.6))
        ax.text(b.x1, b.y1 - 2, f"ID {b.id}", color="white", fontsize=11, fontweight="bold", va="bottom", bbox=dict(fc=c, ec="none", pad=1.5))
    if trajectoires:  # point bas de la boîte, au contact de la route, sur les images précédentes
        for k, g in pistes[(pistes.image <= i) & (pistes.image >= i - 6)].groupby("id"):
            if len(g) >= 2 and i in set(g.image):
                ax.plot((g.x1 + g.x2) / 2, g.y2, "-o", color=couleur_piste(k), lw=2.4, ms=4)


INDICES_AFFICHES = [12, 13, 14]  # mêmes instants que la diapositive 4 : t, t + 0,5 s, t + 1 s
fig, axes = plt.subplots(1, 3, figsize=(20, 4.9))
for k, (ax, i) in enumerate(zip(axes, INDICES_AFFICHES)):
    dessiner_suivi(ax, i, trajectoires=(k == 2))
fig.subplots_adjust(top=0.84, wspace=0.03)
fig.suptitle("Chaque véhicule garde son identifiant d'une image à l'autre (toutes les pistes ; la diapositive n'en montre que 3)",
             x=0.01, ha="left", fontweight="bold")
etiqueter(fig, "reel", "YOLO11n + ByteTrack")
sauvegarder(fig, "suivi_images.png")
plt.show()
presentes = [k for k in pistes.id.unique() if set(INDICES_AFFICHES) <= set(pistes[pistes.id == k].image)]
interpretation(f"{len(presentes)} véhicules portent le même identifiant sur les trois images. "
               "Sans vérité terrain pour cette vidéo, on ne peut pas dire si ces identités sont *correctes* : c'est l'objet de la section 1.3.")

if WIDGETS_OK:  # parcourir toute la vidéo image par image
    explorer(lambda image: dessiner_suivi(plt.subplots(figsize=(8.5, 4.9))[1], image, trajectoires=True),
             image=widgets.IntSlider(value=20, min=0, max=len(images_video) - 1, description="image"))
else:
    print("Installez ipywidgets pour parcourir la vidéo image par image.")

In [ ]:
fig, ax = plt.subplots(figsize=(15, 0.3 * len(duree_pistes) + 1.8))
for k, g in pistes.groupby("id"):
    images_k = np.sort(g.image.unique())
    coupures = np.where(np.diff(images_k) > 1)[0]
    debuts, fins = np.r_[images_k[0], images_k[coupures + 1]], np.r_[images_k[coupures], images_k[-1]]
    for d0, f0 in zip(debuts, fins):  # images consécutives où la piste est détectée
        ax.hlines(k, d0, f0, color=couleur_piste(k), lw=3)
    for f0, d1 in zip(fins[:-1], debuts[1:]):  # interruption : identifiant gardé en mémoire sans détection
        ax.hlines(k, f0, d1, color=PALETTE["gris"], lw=1.2, ls=":")
    ax.scatter(images_k, [k] * len(images_k), s=26, marker="s", color=couleur_piste(k))
ax.legend(handles=[plt.Line2D([], [], color=PALETTE["sarcelle"], lw=3, marker="s", label="piste détectée"),
                   plt.Line2D([], [], color=PALETTE["gris"], lw=1.2, ls=":", label="interruption, même identifiant à la reprise")],
          loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=2, fontsize=11.5, frameon=False)
for i in INDICES_AFFICHES:
    ax.axvline(i, color=PALETTE["gris"], ls=":", lw=1)
ax.set_xlabel("Indice de l'image clé (une toutes les 0,5 s) — pointillés : images affichées ci-dessus")
ax.set_ylabel("Identifiant")
ax.set_yticks(duree_pistes.index)
ax.invert_yaxis()
ax.set_title("Chronologie des identifiants : présence de chaque piste au fil des images", loc="left")
etiqueter(fig, "reel", "YOLO11n + ByteTrack")
sauvegarder(fig, "suivi_chronologie.png")
plt.show()

longues = (duree_pistes.nb_images >= 10).sum()
courtes = (duree_pistes.nb_images < 5).sum()
reprises = sum(1 for _, g in pistes.groupby("id") if (np.diff(np.sort(g.image.values)) > 1).any())
interpretation(f"{longues} pistes durent au moins 10 images (5 s) ; {courtes} pistes durent moins de 5 images (2,5 s). "
               f"{reprises} pistes s'interrompent puis reprennent **avec le même identifiant** : ByteTrack garde une piste en mémoire "
               "quelque temps sans détection avant de l'abandonner. Une piste courte peut être un véhicule qui sort du champ, "
               "une fausse détection ou une trajectoire coupée en deux : seule la vérité terrain permet de trancher.")

### 1.3 Évaluation

$$\text{MOTA} = 1 - \frac{\sum_t (\text{FN}_t + \text{FP}_t + \text{IDSW}_t)}{\sum_t \text{GT}_t}$$

FN : objet réel manqué · FP : fausse alarme · IDSW : changement d'identité · GT : objets réels. Appariement si IoU ≥ 0,5. **Plus haut = meilleur** (maximum 1, peut être négatif).

**Exemple de la diapositive 6** (construit à la main, aucun modèle) :

In [ ]:
def mota(fn, fp, idsw, gt):
    # MOTA = 1 − (FN + FP + IDSW) / GT  (Bernardin et Stiefelhagen, 2008)
    return 1 - (fn + fp + idsw) / gt


GT_P, FN_P, FP_P, IDSW_P = 10, 1, 1, 1
print(f"MOTA = 1 − ({FN_P} + {FP_P} + {IDSW_P}) / {GT_P} = {fmt(mota(FN_P, FP_P, IDSW_P, GT_P))} "
      f"= {fmt(mota(FN_P, FP_P, IDSW_P, GT_P), pct=True)}")

fig, ax = plt.subplots(figsize=(15, 3.1))
ax.set_xlim(-0.2, 15.2)
ax.set_ylim(-0.9, 1.6)
ax.set_axis_off()
etats = ["ok"] * 8 + ["FN", "IDSW"]
for k, e in enumerate(etats):
    if e == "ok":
        ax.add_patch(patches.Rectangle((k, 0), 0.85, 0.85, fc=PALETTE["sauge"], ec="none"))
        ax.text(k + 0.425, 0.425, "✓", ha="center", va="center", color="white", fontsize=18, fontweight="bold")
    elif e == "FN":
        ax.add_patch(patches.Rectangle((k, 0), 0.85, 0.85, fc="white", ec=PALETTE["terracotta"], lw=2, ls="--"))
        ax.text(k + 0.425, 0.425, "FN", ha="center", va="center", color=PALETTE["terracotta"], fontsize=14, fontweight="bold")
    else:
        ax.add_patch(patches.Rectangle((k, 0), 0.85, 0.85, fc=PALETTE["terracotta"], ec="none"))
        ax.text(k + 0.425, 0.425, "IDSW", ha="center", va="center", color="white", fontsize=12, fontweight="bold")
ax.add_patch(patches.Rectangle((10.6, 0), 0.85, 0.85, fc=PALETTE["terracotta"], ec="none"))
ax.text(11.025, 0.425, "FP", ha="center", va="center", color="white", fontsize=14, fontweight="bold")
ax.annotate("", xy=(0, 1.08), xytext=(9.85, 1.08), arrowprops=dict(arrowstyle="<->", color=PALETTE["marine"]))
ax.text(4.9, 1.2, "GT = 10 objets réels", ha="center", fontsize=13)
ax.text(11.025, 1.2, "hors vérité terrain", ha="center", fontsize=12, color=PALETTE["terracotta"])
ax.text(0, -0.5, "FN : objet manqué   ·   IDSW : objet détecté, mais identité échangée   ·   FP : fausse alarme",
        fontsize=12, color=PALETTE["gris"])
ax.text(12.0, 0.42, "MOTA = 1 − 3/10\n= 0,70", fontsize=17, fontweight="bold", color=PALETTE["sarcelle"], va="center")
etiqueter(fig, "pedagogique", "exemple de la diapositive 6")
plt.show()

**Le changement d'identité.** Détection parfaite, mais deux véhicules échangent leurs identités : MOTA le voit à peine, IDF1 beaucoup plus.
$\text{IDF1} = \dfrac{2\,\text{IDTP}}{2\,\text{IDTP} + \text{IDFP} + \text{IDFN}}$ (après appariement global des trajectoires).

In [ ]:
import motmetrics as mm


def evaluer_jouet(sequence):
    # sequence : liste d'images ; chaque image = liste de couples (objet réel, identifiant prédit), détection parfaite
    acc = mm.MOTAccumulator(auto_id=True)
    codes = {o: k + 1 for k, o in enumerate(sorted({o for image in sequence for o, _ in image}))}  # motmetrics attend des nombres
    for image in sequence:
        objets = [codes[o] for o, _ in image]
        predits = [p for _, p in image]
        distances = np.full((len(objets), len(predits)), np.nan)
        np.fill_diagonal(distances, 0.0)  # chaque objet est exactement recouvert par « sa » prédiction
        acc.update(objets, predits, distances)
    r = mm.metrics.create().compute(acc, metrics=["num_objects", "num_switches", "mota", "idf1"], name="x").iloc[0]
    return int(r.num_objects), int(r.num_switches), r.mota, r.idf1


# Exemple 1 (guide) : un objet sur 4 images, identifiant 1 puis 5
ex1 = [[("A", 1)], [("A", 1)], [("A", 5)], [("A", 5)]]
# Exemple 2 (croisement) : deux objets sur 6 images, identités échangées après le croisement
ex2 = [[("A", 1), ("B", 2)]] * 3 + [[("A", 2), ("B", 1)]] * 3
tableau = pd.DataFrame([evaluer_jouet(ex1), evaluer_jouet(ex2)], columns=["GT", "IDSW", "MOTA", "IDF1"],
                       index=["Un objet, identifiant 1 → 5", "Deux objets qui se croisent"])
display(tableau.style.format({"MOTA": "{:.3f}", "IDF1": "{:.3f}"}))

# Schéma du croisement
fig, ax = plt.subplots(figsize=(14, 3.6))
t = np.arange(6)
ya, yb = t / 5, 1 - t / 5
ax.plot(t, ya, "-", color=PALETTE["sarcelle"], lw=2, alpha=0.5)
ax.plot(t, yb, "-", color=PALETTE["terracotta"], lw=2, alpha=0.5)
for k in t:
    id_a, id_b = (1, 2) if k < 3 else (2, 1)
    for y, couleur, ident in [(ya[k], PALETTE["sarcelle"], id_a), (yb[k], PALETTE["terracotta"], id_b)]:
        ax.add_patch(patches.Circle((k, y), 0.13, fc=couleur, ec="white", lw=1.5))
        ax.text(k, y, str(ident), ha="center", va="center", color="white", fontsize=12, fontweight="bold")
ax.annotate("échange d'identités\n(2 IDSW)", xy=(3, 0.6), xytext=(3.45, 1.12), fontsize=12, color=PALETTE["marine"],
            arrowprops=dict(arrowstyle="->", color=PALETTE["marine"]))
ax.text(-0.3, -0.27, "véhicule A (vrai)", color=PALETTE["sarcelle"], fontsize=12, fontweight="bold")
ax.text(-0.3, 1.17, "véhicule B (vrai)", color=PALETTE["terracotta"], fontsize=12, fontweight="bold")
ax.set_xlim(-0.5, 6)
ax.set_ylim(-0.4, 1.35)
ax.set_xticks(t, [f"image {k + 1}" for k in t])
ax.set_yticks([])
ax.spines["left"].set_visible(False)
ax.set_title("La couleur = l'objet réel ; le chiffre = l'identifiant prédit. Détection parfaite, identités échangées.", loc="left")
etiqueter(fig, "conceptuel")
plt.show()
interpretation(f"Avec une détection parfaite, MOTA reste à {fmt(tableau.MOTA.iloc[1], 3)} (2 erreurs sur 12 objets réels), "
               f"mais IDF1 tombe à {fmt(tableau.IDF1.iloc[1])} : la moitié du temps, chaque véhicule porte l'identifiant de l'autre.")

**Mesure réelle sur MOT17-04** (8 images, vérité terrain officielle, protocole MOTChallenge : piétons « à évaluer », distracteurs ignorés, IoU ≥ 0,5). La vidéo nuScenes n'a pas d'annotations de suivi 2D : on ne compare jamais le modèle à ses propres prédictions. *8 images illustrent le calcul ; ce n'est pas un benchmark.*

In [ ]:
from scipy.optimize import linear_sum_assignment

SEQ_MOT = ASSETS / "suivi" / "MOT17-04-FRCNN_extrait"
gt_mot = pd.read_csv(SEQ_MOT / "gt" / "gt.txt", header=None,
                     names=["image", "id", "x", "y", "l", "h", "a_evaluer", "classe", "visibilite"])
CLASSES_DISTRACTEURS = [2, 7, 8, 12]  # personne sur un véhicule, personne statique, distracteur, reflet
images_mot = sorted((SEQ_MOT / "img1").glob("*.jpg"))
gt_eval = gt_mot[(gt_mot.a_evaluer == 1) & (gt_mot.classe == 1)]
print(f"Extrait MOT17-04 : {len(images_mot)} images · {len(gt_eval)} boîtes de piétons à évaluer "
      f"({gt_eval.id.nunique()} personnes) · {gt_mot.classe.isin(CLASSES_DISTRACTEURS).sum()} boîtes de distracteurs")


def matrice_iou(a, b):
    # IoU entre deux ensembles de boîtes au format (x, y, largeur, hauteur)
    a = np.asarray(a, dtype=float).reshape(-1, 4)
    b = np.asarray(b, dtype=float).reshape(-1, 4)
    lx = np.clip(np.minimum(a[:, None, 0] + a[:, None, 2], b[None, :, 0] + b[None, :, 2]) - np.maximum(a[:, None, 0], b[None, :, 0]), 0, None)
    ly = np.clip(np.minimum(a[:, None, 1] + a[:, None, 3], b[None, :, 1] + b[None, :, 3]) - np.maximum(a[:, None, 1], b[None, :, 1]), 0, None)
    inter = lx * ly
    return inter / (a[:, None, 2] * a[:, None, 3] + b[None, :, 2] * b[None, :, 3] - inter)


def suivre_sequence(chemins, tracker="bytetrack.yaml", taille=1280, conf=0.1):
    # Détection des personnes (classe COCO 0) + suivi ; une nouvelle instance du modèle remet le suivi à zéro
    modele = YOLO(str(MODELES / "yolo11n.pt"))
    lignes = []
    for f, chemin in enumerate(chemins, start=1):
        res = modele.track(str(chemin), persist=True, tracker=tracker, classes=[0], conf=conf, imgsz=taille,
                           device=DEVICE_YOLO, verbose=False)[0]
        if res.boxes.id is None:
            continue
        for tid, (x1, y1, x2, y2) in zip(res.boxes.id.int().tolist(), res.boxes.xyxy.tolist()):
            lignes.append({"image": f, "id": tid, "x": x1, "y": y1, "l": x2 - x1, "h": y2 - y1})
    return pd.DataFrame(lignes, columns=["image", "id", "x", "y", "l", "h"])


def evaluer_suivi(gt, pred, seuil_iou=0.5):
    acc = mm.MOTAccumulator(auto_id=False)
    acc.donnees_hota = []  # (identifiants réels, identifiants prédits, matrice IoU) par image, pour HOTA
    for f in sorted(gt.image.unique()):
        g = gt[gt.image == f]
        g_eval = g[(g.a_evaluer == 1) & (g.classe == 1)]
        g_distr = g[g.classe.isin(CLASSES_DISTRACTEURS)]
        p = pred[pred.image == f]
        if len(g_distr) and len(p):  # protocole MOTChallenge : on retire les prédictions appariées à un distracteur
            iou_d = matrice_iou(p[["x", "y", "l", "h"]].values, g_distr[["x", "y", "l", "h"]].values)
            lig, col = linear_sum_assignment(-iou_d)
            p = p.drop(index=[p.index[i] for i, j in zip(lig, col) if iou_d[i, j] >= seuil_iou])
        iou_e = matrice_iou(g_eval[["x", "y", "l", "h"]].values, p[["x", "y", "l", "h"]].values)
        acc.update(g_eval.id.values, p.id.values, np.where(iou_e >= seuil_iou, 1 - iou_e, np.nan), frameid=int(f))
        acc.donnees_hota.append((g_eval.id.values, p.id.values, iou_e))
    noms = ["num_frames", "num_objects", "num_predictions", "num_matches", "num_misses", "num_false_positives",
            "num_switches", "mota", "motp", "idf1", "precision", "recall"]
    return mm.metrics.create().compute(acc, metrics=noms, name="résultat").iloc[0], acc


with chrono("YOLO11n + ByteTrack sur l'extrait MOT17"):
    pred_mot = suivre_sequence(images_mot)
res_mot, acc_mot = evaluer_suivi(gt_mot, pred_mot)

verif = 1 - (res_mot.num_misses + res_mot.num_false_positives + res_mot.num_switches) / res_mot.num_objects
tableau_mot = pd.Series({
    "Images": int(res_mot.num_frames), "Objets réels (GT)": int(res_mot.num_objects),
    "Prédictions évaluées": int(res_mot.num_predictions), "Appariements": int(res_mot.num_matches),
    "FN (manqués)": int(res_mot.num_misses), "FP (fausses alarmes)": int(res_mot.num_false_positives),
    "IDSW": int(res_mot.num_switches), "MOTA": round(res_mot.mota, 3), "IDF1": round(res_mot.idf1, 3),
    "Précision": round(res_mot.precision, 3), "Rappel": round(res_mot.recall, 3),
    "IoU moyen des appariements": round(1 - res_mot.motp, 3),
}, name="YOLO11n + ByteTrack").to_frame()
display(tableau_mot)
print(f"Vérification à la main : 1 − ({int(res_mot.num_misses)} + {int(res_mot.num_false_positives)} + "
      f"{int(res_mot.num_switches)}) / {int(res_mot.num_objects)} = {fmt(verif, 3)} (motmetrics : {fmt(res_mot.mota, 3)})")
RESULTATS["suivi"] = {"MOTA": res_mot.mota, "IDF1": res_mot.idf1}

In [ ]:
def hota(donnees, alphas=np.arange(0.05, 0.99, 0.05)):
    # HOTA (Luiten et al., 2021), même algorithme que TrackEval : appariement par image pondéré par l'alignement global
    # des trajectoires, puis DetA et AssA pour chaque seuil de localisation α ; HOTA = moyenne sur les 19 seuils.
    ids_g = {k: i for i, k in enumerate(sorted({int(x) for g, _, _ in donnees for x in g}))}
    ids_p = {k: i for i, k in enumerate(sorted({int(x) for _, p, _ in donnees for x in p}))}
    potentiel = np.zeros((len(ids_g), len(ids_p)))
    n_g, n_p = np.zeros((len(ids_g), 1)), np.zeros((1, len(ids_p)))
    trames = []
    for g, p, sim in donnees:
        g_i, p_i = np.array([ids_g[int(x)] for x in g], dtype=int), np.array([ids_p[int(x)] for x in p], dtype=int)
        sim = np.asarray(sim, dtype=float).reshape(len(g_i), len(p_i))
        trames.append((g_i, p_i, sim))
        if len(g_i) and len(p_i):
            denom = sim.sum(0)[None, :] + sim.sum(1)[:, None] - sim
            potentiel[g_i[:, None], p_i[None, :]] += np.divide(sim, denom, out=np.zeros_like(sim), where=denom > np.finfo(float).eps)
        n_g[g_i] += 1
        n_p[0, p_i] += 1
    alignement = potentiel / (n_g + n_p - potentiel)
    TP, FN, FP = np.zeros(len(alphas)), np.zeros(len(alphas)), np.zeros(len(alphas))
    appariements = [np.zeros_like(potentiel) for _ in alphas]
    for g_i, p_i, sim in trames:
        if len(g_i) == 0 or len(p_i) == 0:
            FN += len(g_i)
            FP += len(p_i)
            continue
        lig, col = linear_sum_assignment(-(alignement[g_i[:, None], p_i[None, :]] * sim))
        for a, alpha in enumerate(alphas):
            ok = sim[lig, col] >= alpha - np.finfo(float).eps
            TP[a] += ok.sum()
            FN[a] += len(g_i) - ok.sum()
            FP[a] += len(p_i) - ok.sum()
            appariements[a][g_i[lig[ok]], p_i[col[ok]]] += 1
    AssA = np.array([np.sum(m * m / np.maximum(1, n_g + n_p - m)) / max(1, tp) for m, tp in zip(appariements, TP)])
    DetA = TP / np.maximum(1, TP + FN + FP)
    HOTA_a = np.sqrt(DetA * AssA)
    return {"HOTA": HOTA_a.mean(), "DetA": DetA.mean(), "AssA": AssA.mean(), "alphas": alphas, "HOTA_a": HOTA_a, "DetA_a": DetA, "AssA_a": AssA}


h = hota(acc_mot.donnees_hota)
RESULTATS["suivi"]["HOTA"] = h["HOTA"]
display(pd.Series({"MOTA": res_mot.mota, "IDF1": res_mot.idf1, "HOTA": h["HOTA"], "DetA": h["DetA"], "AssA": h["AssA"]},
                  name="YOLO11n + ByteTrack").to_frame().T.style.format("{:.3f}"))

fig, ax = plt.subplots(figsize=(12, 4.3))
for cle, couleur, nom in [("DetA_a", PALETTE["ocre"], "DetA : qualité de détection"), ("AssA_a", PALETTE["sauge"], "AssA : qualité d'association"),
                          ("HOTA_a", PALETTE["sarcelle"], "HOTA = √(DetA · AssA)")]:
    ax.plot(h["alphas"], h[cle], "-o", color=couleur, lw=2.5 if cle == "HOTA_a" else 1.8, ms=5, label=nom)
ax.set_xlabel("Seuil de localisation α (IoU minimal pour compter un appariement)")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
ax.set_title(f"HOTA en fonction du seuil α : moyenne sur 19 seuils = {fmt(h['HOTA'], 3)}", loc="left")
etiqueter(fig, "reel", "MOT17-04, 8 images")
plt.show()
interpretation(f"HOTA = {fmt(h['HOTA'], 3)} : la détection (DetA = {fmt(h['DetA'], 3)}) est le point faible, l'association "
               f"(AssA = {fmt(h['AssA'], 3)}) est bonne sur une séquence aussi courte. Plus le seuil α est exigeant, plus DetA chute : "
               "HOTA récompense aussi la **précision des boîtes**, ce que MOTA ignore au-delà du seuil de 0,5.")

In [ ]:
# Où sont les erreurs ? Visualisation de la dernière image : appariements, objets manqués, fausses alarmes
f_vis = int(gt_mot.image.max())
ev = acc_mot.mot_events.loc[f_vis]
oid_ok = set(ev[ev.Type.isin(["MATCH", "SWITCH"])].OId)
hid_ok = set(ev[ev.Type.isin(["MATCH", "SWITCH"])].HId)
hid_fp = set(ev[ev.Type == "FP"].HId)
g = gt_eval[gt_eval.image == f_vis]
p = pred_mot[pred_mot.image == f_vis]

fig, ax = plt.subplots(figsize=(16, 9.4))
ax.imshow(Image.open(images_mot[f_vis - 1]))
ax.set_axis_off()
for _, b in g.iterrows():
    if b.id not in oid_ok:  # objet réel manqué
        ax.add_patch(patches.Rectangle((b.x, b.y), b.l, b.h, fill=False, ec=PALETTE["terracotta"], lw=2.4, ls="--"))
for _, b in p.iterrows():
    couleur = PALETTE["sauge"] if b.id in hid_ok else (PALETTE["terracotta"] if b.id in hid_fp else PALETTE["gris"])
    ax.add_patch(patches.Rectangle((b.x, b.y), b.l, b.h, fill=False, ec=couleur, lw=2.2))
legende = [patches.Patch(fc="none", ec=PALETTE["sauge"], lw=2, label=f"prédiction appariée (IoU ≥ 0,5) : {len(oid_ok)}"),
           patches.Patch(fc="none", ec=PALETTE["terracotta"], lw=2, ls="--", label=f"objet réel manqué (FN) : {len(g) - len(oid_ok)}"),
           patches.Patch(fc="none", ec=PALETTE["terracotta"], lw=2, label=f"fausse alarme (FP) : {len(hid_fp)}"),
           patches.Patch(fc="none", ec=PALETTE["gris"], lw=2, label="prédiction sur un distracteur (ignorée)")]
ax.legend(handles=legende, loc="lower left", framealpha=0.92, fontsize=12)
ax.set_title(f"MOT17-04, image {f_vis} : prédictions de YOLO11n + ByteTrack comparées à la vérité terrain", loc="left")
etiqueter(fig, "reel", "comparaison à la vérité terrain")
sauvegarder(fig, "suivi_mot17_erreurs.png")
plt.show()

fig, ax = plt.subplots(figsize=(11, 2.9))
valeurs = [res_mot.num_misses, res_mot.num_false_positives, res_mot.num_switches]
noms = ["FN (manqués)", "FP (fausses alarmes)", "IDSW (identités)"]
ax.barh(noms, valeurs, color=[PALETTE["terracotta"], PALETTE["ocre"], PALETTE["marine"]])
for k, v in enumerate(valeurs):
    ax.text(v + 1, k, f"{int(v)}  ({fmt(v / res_mot.num_objects, pct=True)} des objets réels)", va="center", fontsize=12)
ax.invert_yaxis()
ax.set_xlim(0, max(valeurs) * 1.6 + 1)
ax.set_xlabel("Nombre d'erreurs sur les 8 images")
ax.set_title(f"Décomposition des erreurs : MOTA = 1 − {int(sum(valeurs))}/{int(res_mot.num_objects)} = {fmt(res_mot.mota, 3)}", loc="left")
etiqueter(fig, "reel", "MOT17-04, 8 images")
plt.show()
interpretation(f"MOTA = {fmt(res_mot.mota, 3)} et IDF1 = {fmt(res_mot.idf1, 3)}. Les objets **manqués** dominent : "
               "beaucoup de piétons sont petits et lointains, et YOLO11n est le plus petit détecteur de sa famille. "
               f"Aucun changement d'identité n'apparaît sur 0,27 s ({int(res_mot.num_switches)} IDSW) : "
               "c'est trop court pour juger la stabilité des identités, pas la preuve d'un suivi parfait.")

**Effet du détecteur** : même suivi, images réduites à 640 px au lieu de 1 280 px.

In [ ]:
with chrono("Suivi à 640 px"):
    res_640, _ = evaluer_suivi(gt_mot, suivre_sequence(images_mot, taille=640))
comparaison = pd.DataFrame({"entrée 640 px": res_640, "entrée 1 280 px": res_mot}).T[
    ["num_misses", "num_false_positives", "num_switches", "mota", "idf1"]]
comparaison.columns = ["FN", "FP", "IDSW", "MOTA", "IDF1"]
display(comparaison.style.format({"FN": "{:.0f}", "FP": "{:.0f}", "IDSW": "{:.0f}", "MOTA": "{:.3f}", "IDF1": "{:.3f}"}))
interpretation("Réduire la résolution fait exploser le nombre de piétons manqués : MOTA mesure d'abord la qualité de la **détection**. "
               "Comparer deux méthodes de suivi n'a de sens qu'avec le même détecteur et le même protocole.")

In [ ]:
# Mode B : comparaison de deux algorithmes d'association avec le même détecteur
if MODE == "B":
    lignes_comp = {}
    for nom, fichier in [("ByteTrack", "bytetrack.yaml"), ("BoT-SORT", "botsort.yaml")]:
        r, _ = evaluer_suivi(gt_mot, suivre_sequence(images_mot, tracker=fichier))
        lignes_comp[nom] = r[["num_misses", "num_false_positives", "num_switches", "mota", "idf1"]]
    comp = pd.DataFrame(lignes_comp).T
    comp.columns = ["FN", "FP", "IDSW", "MOTA", "IDF1"]
    display(comp.style.format({"FN": "{:.0f}", "FP": "{:.0f}", "IDSW": "{:.0f}", "MOTA": "{:.3f}", "IDF1": "{:.3f}"}))
    interpretation("Avec le même détecteur, les deux algorithmes d'association donnent des scores très proches sur 8 images : "
                   "une différence de quelques centièmes n'est pas significative sur un extrait aussi court.")
else:
    print("Comparaison ByteTrack / BoT-SORT : disponible en mode B.")

### Conclusion
MOTA est dominé par la **détection** (objets manqués, fausses alarmes) ; IDF1 vérifie la **stabilité des identités** ; HOTA équilibre les deux et tient compte de la précision des boîtes. Un score n'a de sens qu'avec son protocole : seuil d'IoU, classes évaluées, distracteurs, longueur de la séquence.

---
# 2. Génération d'images — diapositives 7 à 9

**Objectif.** Générer une image à partir du prompt de la diapositive 7, observer les vraies étapes du débruitage, puis comprendre ses deux évaluations : **FID** (réalisme d'un ensemble d'images) et **CLIPScore** (respect du texte, image par image).

**Modèle** : Stable Diffusion (mode B, GPU) ; CLIP pour l'accord texte-image. **Pas de vérité terrain unique** : on compare des distributions.

In [ ]:
schema_pipeline(["Prompt", "Encodeur texte", "Débruitage itératif", "Décodeur VAE", "Image"],
                "Chaîne de génération d'une diffusion latente (Stable Diffusion)",
                ["texte libre", "vecteurs (CLIP)", "U-Net, T étapes", "latent → pixels", "512 × 512"], mise_en_avant=2)

### 2.1 Le principe, sur une vraie photo (diapositive 8)
Bruitage **exact** $x_t = \sqrt{\bar\alpha_t}\,x_0 + \sqrt{1-\bar\alpha_t}\,\varepsilon$, affiché à rebours : c'est le chemin que le modèle apprend à parcourir. Aucun modèle n'est exécuté ici.

In [ ]:
T_DIFFUSION = 1000
betas = np.linspace(1e-4, 0.02, T_DIFFUSION)
alpha_barre = np.cumprod(1 - betas)  # alpha_barre[t-1] correspond à l'étape t

photo_cafe = Image.open(ASSETS / "generation" / "cafe_scikit_image.jpg").convert("RGB")
x0 = np.asarray(photo_cafe.resize((256, 170)), dtype=np.float32) / 127.5 - 1  # pixels ramenés dans [-1, 1]
bruit = np.random.default_rng(GRAINE).standard_normal(x0.shape).astype(np.float32)


def bruiter(t):
    # Processus direct q(x_t | x_0) : formule exacte, sans aucun réseau de neurones
    if t == 0:
        return x0
    a = alpha_barre[t - 1]
    return np.sqrt(a) * x0 + np.sqrt(1 - a) * bruit


etapes = [1000, 600, 400, 200, 0]
fig = plt.figure(figsize=(18, 5.6))
grille = fig.add_gridspec(2, 5, height_ratios=[3, 2], hspace=0.45)
for k, t in enumerate(etapes):
    ax = fig.add_subplot(grille[0, k])
    ax.imshow(np.clip((bruiter(t) + 1) / 2, 0, 1))
    ax.set_axis_off()
    signal = 1.0 if t == 0 else np.sqrt(alpha_barre[t - 1])
    ax.set_title(f"t = {t}\npart du signal : {fmt(signal)}", fontsize=12)
ax = fig.add_subplot(grille[1, :])
pas = np.arange(1, T_DIFFUSION + 1)
ax.plot(pas, np.sqrt(alpha_barre), color=PALETTE["sarcelle"], lw=2.5, label="√ᾱₜ : poids de l'image")
ax.plot(pas, np.sqrt(1 - alpha_barre), color=PALETTE["terracotta"], lw=2.5, label="√(1 − ᾱₜ) : poids du bruit")
ax.set_xlabel("Étape de bruitage t")
ax.set_ylabel("Coefficient")
ax.invert_xaxis()
ax.legend(loc="center left")
fig.suptitle("Du bruit pur à l'image : bruitage exact d'une photo réelle, affiché à rebours", x=0.01, ha="left", fontweight="bold")
etiqueter(fig, "conceptuel", "aucun modèle exécuté")
plt.show()

### 2.2 Entrée : prompt et photo de référence
Le prompt français est traduit en anglais (langue d'entraînement de Stable Diffusion) ; la traduction est affichée. La photo réelle sert de **point de comparaison** : ce n'est pas une image générée.

In [ ]:
PROMPT_FR = "Photo produit : un expresso dans une tasse rouge, sur une table en bois"
PROMPT_EN = "Product photo: an espresso in a red cup on a wooden table"  # traduction utilisée par le modèle
print(f"Prompt de la diapositive : {PROMPT_FR}")
print(f"Prompt envoyé au modèle   : {PROMPT_EN}")

fig, ax = plt.subplots(figsize=(6.5, 4.6))
ax.imshow(photo_cafe)
ax.set_axis_off()
ax.set_title("Photo réelle de la diapositive 7 (non générée)", fontsize=13)
etiqueter(fig, "donnees")
plt.show()

### 2.3 Prédiction : génération réelle (mode B)
GPU + mode B : Stable Diffusion 1.5, 30 étapes (≈ 3 Go à télécharger). `GENERER_IMAGE_SUR_CPU = True` : SD-Turbo, 2 étapes (≈ 5 Go, plusieurs minutes). Sinon : génération ignorée, **aucune image n'est simulée**.

In [ ]:
GENERATION_ACTIVE = (MODE == "B" and DEVICE in ("cuda", "mps")) or GENERER_IMAGE_SUR_CPU
image_generee, etats_intermediaires, config_generation, pipe = None, {}, None, None

if GENERATION_ACTIVE:
    try:
        from diffusers import AutoPipelineForText2Image
    except ImportError:
        GENERATION_ACTIVE = False
        print("La bibliothèque « diffusers » est absente : pip install diffusers accelerate")

if GENERATION_ACTIVE:
    candidats = [{"id": "sd-legacy/stable-diffusion-v1-5", "etapes": 30, "guidage": 7.5},
                 {"id": "stable-diffusion-v1-5/stable-diffusion-v1-5", "etapes": 30, "guidage": 7.5},
                 {"id": "stabilityai/sd-turbo", "etapes": 4, "guidage": 0.0}]
    if DEVICE == "cpu":  # sur CPU, le modèle distillé en très peu d'étapes est le seul raisonnable
        candidats = [{"id": "stabilityai/sd-turbo", "etapes": 2, "guidage": 0.0}] + candidats[:2]
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    for candidat in candidats:
        for variante in (["fp16", None] if DEVICE == "cuda" else [None]):
            try:
                with chrono(f"Chargement de {candidat['id']}"):
                    pipe = AutoPipelineForText2Image.from_pretrained(candidat["id"], torch_dtype=dtype, variant=variante)
                pipe = pipe.to(DEVICE)
                pipe.set_progress_bar_config(disable=True)
                config_generation = candidat
                break
            except Exception as erreur:
                print(f"  {candidat['id']} (variante {variante}) indisponible : {type(erreur).__name__}")
        if config_generation:
            break

if config_generation:
    def capturer(pipe_, indice, pas_de_temps, variables):
        # Appelée après chaque étape de débruitage : on copie le latent courant (64 × 64 × 4)
        etats_intermediaires[indice + 1] = variables["latents"].detach().to("cpu", torch.float32).clone()
        return variables

    generateur = torch.Generator(device="cpu").manual_seed(GRAINE)
    latent_initial = torch.randn((1, 4, 64, 64), generator=generateur, dtype=torch.float32)
    etats_intermediaires[0] = latent_initial.clone()
    with chrono(f"Génération ({config_generation['etapes']} étapes)"):
        image_generee = pipe(PROMPT_EN, num_inference_steps=config_generation["etapes"],
                             guidance_scale=config_generation["guidage"], latents=latent_initial.to(DEVICE, dtype),
                             callback_on_step_end=capturer, callback_on_step_end_tensor_inputs=["latents"]).images[0]
    image_generee.save(SORTIES / "image_generee.png")
    print(f"Image générée par {config_generation['id']} (graine {GRAINE}) : resultats/image_generee.png")
else:
    message = ("GPU détecté : passez MODE = \"B\" pour générer l'image." if DEVICE != "cpu" else
               "Pas de GPU : génération ignorée (mettre GENERER_IMAGE_SUR_CPU = True pour la tenter sur CPU).")
    print(message + " Aucune image générée n'est simulée ni remplacée.")

In [ ]:
def decoder_latent(latent):
    # Décode un latent (64 × 64 × 4) en image 512 × 512 avec le VAE du pipeline
    with torch.no_grad():
        x = latent.to(pipe.device, dtype=pipe.vae.dtype) / pipe.vae.config.scaling_factor
        image = pipe.vae.decode(x, return_dict=False)[0]
    return pipe.image_processor.postprocess(image.float(), output_type="pil")[0]


if image_generee is not None:
    n = config_generation["etapes"]
    choix = sorted(set([0, 1] + [round(n * f) for f in (0.2, 0.4, 0.6, 0.8)] + [n]))
    fig, axes = plt.subplots(1, len(choix), figsize=(3.1 * len(choix), 4.2))
    for ax, k in zip(axes, choix):
        ax.imshow(image_generee if k == n else decoder_latent(etats_intermediaires[k]))
        ax.set_axis_off()
        ax.set_title("bruit initial" if k == 0 else (f"étape {k}/{n}\n(image finale)" if k == n else f"étape {k}/{n}"), fontsize=12)
    fig.suptitle(f"« {PROMPT_FR} » — vrais états du débruitage, décodés par le VAE", x=0.01, ha="left", fontweight="bold")
    etiqueter(fig, "reel", config_generation["id"])
    sauvegarder(fig, "generation_etapes.png")
    plt.show()
    interpretation("Les premières étapes fixent la composition et les couleurs ; les dernières affinent les détails. "
                   "Chaque vignette est le véritable état du modèle à cette étape, pas une illustration.")
else:
    print("Pas d'image générée dans cette exécution : voir la section 2.3.")

### 2.4 Évaluation : FID (diapositive 9)

$$\text{FID} = \lVert \mu_r - \mu_g \rVert^2 + \operatorname{Tr}\big(\Sigma_r + \Sigma_g - 2(\Sigma_r\Sigma_g)^{1/2}\big)$$

$\mu$, $\Sigma$ : moyenne et covariance des caractéristiques Inception (2 048 dimensions) des images réelles ($r$) et générées ($g$). **Plus bas = meilleur.** En une dimension : $(\mu_r-\mu_g)^2 + (\sigma_r-\sigma_g)^2$.

In [ ]:
def racine_symetrique(M):
    # Racine carrée d'une matrice symétrique positive, par décomposition en valeurs propres
    valeurs, vecteurs = np.linalg.eigh((M + M.T) / 2)
    return (vecteurs * np.sqrt(np.clip(valeurs, 0, None))) @ vecteurs.T


def distance_frechet(mu1, sigma1, mu2, sigma2):
    # FID = ||mu1 - mu2||² + Tr(S1 + S2 - 2 (S1 S2)^½)  (Heusel et al., 2017)
    # Identité utilisée pour la stabilité numérique : Tr((S1 S2)^½) = Tr((S1^½ S2 S1^½)^½)
    r1 = racine_symetrique(sigma1)
    trace_racine = np.trace(racine_symetrique(r1 @ sigma2 @ r1))
    return float(np.sum((mu1 - mu2) ** 2) + np.trace(sigma1) + np.trace(sigma2) - 2 * trace_racine)


# Exemple de la diapositive 9 : réelles (μ = 0, σ = 1), générées (μ = 1, σ = 2) → (0 − 1)² + (1 − 2)² = 2
print("Exemple de la diapositive 9 (1 dimension) : FID =",
      fmt(distance_frechet(np.array([0.0]), np.array([[1.0 ** 2]]), np.array([1.0]), np.array([[2.0 ** 2]]))), "(attendu : 2)")

mu_r, sigma_r = np.array([0.0, 0.0]), np.array([[1.0, 0.35], [0.35, 0.6]])
scenarios = [("Décalée : même forme, centre déplacé", np.array([1.2, 0.5]), sigma_r),
             ("Peu diverse : même centre, nuage trop concentré", np.array([0.0, 0.0]), 0.15 * sigma_r),
             ("Proche des images réelles", np.array([0.1, -0.05]), 1.05 * sigma_r)]
gen = np.random.default_rng(GRAINE)


def ellipse(ax, mu, sigma, couleur):
    valeurs, vecteurs = np.linalg.eigh(sigma)
    angle = np.degrees(np.arctan2(vecteurs[1, 1], vecteurs[0, 1]))
    ax.add_patch(patches.Ellipse(mu, 4 * np.sqrt(valeurs[1]), 4 * np.sqrt(valeurs[0]), angle=angle, fill=False, ec=couleur, lw=2))


fig, axes = plt.subplots(1, 3, figsize=(18, 5.4), sharex=True, sharey=True)
reelles = gen.multivariate_normal(mu_r, sigma_r, 300)
for ax, (titre, mu_g, sigma_g) in zip(axes, scenarios):
    generees = gen.multivariate_normal(mu_g, sigma_g, 300)
    ax.scatter(*reelles.T, s=9, color=PALETTE["sarcelle"], alpha=0.45, label="images réelles")
    ax.scatter(*generees.T, s=9, color=PALETTE["terracotta"], alpha=0.45, label="images générées")
    ellipse(ax, mu_r, sigma_r, PALETTE["sarcelle"])
    ellipse(ax, mu_g, sigma_g, PALETTE["terracotta"])
    ax.plot(*mu_r, "P", color=PALETTE["marine"], ms=11)
    ax.plot(*mu_g, "X", color=PALETTE["marine"], ms=11)
    ax.set_title(f"{titre}\nFID = {fmt(distance_frechet(mu_r, sigma_r, mu_g, sigma_g), 3)}", fontsize=13)
    ax.set_xlabel("caractéristique 1")
axes[0].set_ylabel("caractéristique 2")
axes[0].legend(loc="upper left", fontsize=11)
fig.suptitle("FID en 2 dimensions : un décalage du centre ou un manque de diversité augmentent la distance",
             x=0.01, ha="left", fontweight="bold")
fig.subplots_adjust(top=0.8)
etiqueter(fig, "pedagogique", "distributions simulées")
plt.show()
interpretation("Le scénario « peu divers » a le même centre que les images réelles : seul le terme de trace le pénalise. "
               "C'est ce terme qui détecte un générateur qui produit toujours la même image.")

**Pourquoi aucun FID n'est calculé sur l'image générée.** Avec une image, la covariance n'existe pas ; même avec des centaines d'images, l'estimation est biaisée. Simulation : deux échantillons de **la même** distribution (vrai FID = 0).

In [ ]:
d = 64
A = np.random.default_rng(1).standard_normal((d, d))
sigma_vraie = A @ A.T / d + 0.1 * np.eye(d)
tailles = [20, 50, 100, 300, 1000, 3000, 10000]
moyennes, ecarts = [], []
for n in tailles:
    estimations = []
    for rep in range(5):
        g = np.random.default_rng(100 * n + rep)
        e1, e2 = g.multivariate_normal(np.zeros(d), sigma_vraie, n), g.multivariate_normal(np.zeros(d), sigma_vraie, n)
        estimations.append(distance_frechet(e1.mean(0), np.cov(e1, rowvar=False), e2.mean(0), np.cov(e2, rowvar=False)))
    moyennes.append(np.mean(estimations))
    ecarts.append(np.std(estimations))

fig, ax = plt.subplots(figsize=(12, 4.6))
ax.errorbar(tailles, moyennes, yerr=ecarts, fmt="o-", color=PALETTE["sarcelle"], lw=2.2, capsize=5, ms=7,
            label="FID estimé (moyenne ± écart-type sur 5 tirages)")
ax.axhline(0, color=PALETTE["terracotta"], ls="--", lw=2, label="vrai FID : 0 (même distribution)")
ax.set_xscale("log")
ax.set_xlabel("Nombre d'images par ensemble (échelle logarithmique)")
ax.set_ylabel("FID estimé")
ax.set_title("Deux échantillons de la même distribution (64 dimensions) : le FID estimé n'est nul que pour de très grands ensembles", loc="left", fontsize=13)
ax.legend()
etiqueter(fig, "pedagogique", "données simulées")
plt.show()
interpretation(f"Avec 20 images, le FID estimé vaut environ {fmt(moyennes[0], 1)} alors que les deux ensembles viennent de la même distribution ; "
               f"il tombe à {fmt(moyennes[-1], 2)} avec 10 000 images. Un FID ne se compare qu'à nombre d'images égal.")

### 2.5 Évaluation : accord texte-image avec CLIP
$\text{CLIPScore} = 2{,}5 \cdot \max(\cos(E_I, E_T), 0)$ avec CLIP ViT-B/32 d'OpenAI (Hessel et al., 2021). Si ce modèle est inaccessible, un modèle de repli (LAION-400M) est utilisé et le score est alors signalé comme « de type CLIPScore ».

In [ ]:
with chrono("Chargement de CLIP"):
    CLIP = charger_clip()
NOM_SCORE = "CLIPScore" if CLIP["clipscore_strict"] else "score de type CLIPScore"
print(f"Modèle utilisé : {CLIP['nom']}")

In [ ]:
prompts = {"prompt anglais (traduit)": PROMPT_EN, "prompt français (original)": PROMPT_FR,
           "témoin : chat": "A cat sleeping on a sofa",
           "témoin : cycliste": "A cyclist crossing a street in the rain"}
images_eval = {"photo réelle (diapositive)": photo_cafe}
if image_generee is not None:
    images_eval["image générée"] = image_generee

vecteurs_textes = encoder_textes(CLIP, list(prompts.values()))
vecteurs_images = encoder_images(CLIP, list(images_eval.values()))
cosinus = (vecteurs_images @ vecteurs_textes.T).numpy()
tableau_clip = pd.DataFrame(cosinus, index=list(images_eval), columns=list(prompts))
display(Markdown(f"**Similarité cosinus brute** ({CLIP['nom']})"))
display(tableau_clip.style.format("{:.3f}"))
display(Markdown(f"**{NOM_SCORE} = 2,5 × max(cos θ, 0)**"))
display((2.5 * tableau_clip.clip(lower=0)).style.format("{:.3f}"))

fig, axes = plt.subplots(1, len(images_eval) + 1, figsize=(6 * len(images_eval) + 10, 4.8),
                         gridspec_kw={"width_ratios": [1] * len(images_eval) + [1.5], "wspace": 0.55})
for ax, (nom, im) in zip(axes, images_eval.items()):
    ax.imshow(im)
    ax.set_axis_off()
    ax.set_title(nom, fontsize=13)
ax = axes[-1]
largeur = 0.8 / len(images_eval)
couleurs = [PALETTE["sarcelle"], PALETTE["ocre"]]
for k, nom in enumerate(images_eval):
    valeurs = 2.5 * np.clip(tableau_clip.loc[nom].values, 0, None)
    ax.barh(np.arange(len(prompts)) + k * largeur, valeurs, height=largeur, color=couleurs[k], label=nom)
ax.set_yticks(np.arange(len(prompts)) + largeur * (len(images_eval) - 1) / 2, list(prompts))
ax.invert_yaxis()
ax.set_xlabel(NOM_SCORE)
ax.legend(loc="lower right")
ax.set_title("Accord texte-image : le bon prompt doit se détacher des témoins", fontsize=13, loc="left")
etiqueter(fig, "reel", CLIP["nom"])
sauvegarder(fig, "generation_clip.png")
plt.show()

meilleur = tableau_clip.loc["photo réelle (diapositive)"].idxmax()
interpretation(f"Pour la photo réelle, le texte le plus proche est « {meilleur} ». L'écart avec les témoins compte davantage "
               "que la valeur absolue : même une image parfaitement adaptée donne un cosinus bien inférieur à 1 avec CLIP. "
               "Le prompt français obtient un score différent du prompt anglais : CLIP a été entraîné surtout sur de l'anglais.")
RESULTATS["generation"] = {"cosinus_photo": float(tableau_clip.loc["photo réelle (diapositive)", "prompt anglais (traduit)"]),
                           "image_generee": image_generee is not None, "modele_clip": CLIP["nom"]}
if image_generee is not None:
    RESULTATS["generation"]["clipscore_image_generee"] = float(2.5 * max(tableau_clip.loc["image générée", "prompt anglais (traduit)"], 0))

In [ ]:
# Mode B : plusieurs graines pour le même prompt (variabilité du score), sans prétendre calculer un FID
if image_generee is not None and MODE == "B":
    graines = [1, 2, 3, 4]
    images_graines = []
    with chrono(f"Génération de {len(graines)} images supplémentaires"):
        for gr in graines:
            images_graines.append(pipe(PROMPT_EN, num_inference_steps=config_generation["etapes"],
                                       guidance_scale=config_generation["guidage"],
                                       generator=torch.Generator(device="cpu").manual_seed(gr)).images[0])
    scores = 2.5 * np.clip((encoder_images(CLIP, images_graines) @ encoder_textes(CLIP, [PROMPT_EN]).T).numpy().ravel(), 0, None)
    fig, axes = plt.subplots(1, len(graines), figsize=(4.4 * len(graines), 4.8))
    for ax, im, gr, s in zip(axes, images_graines, graines, scores):
        ax.imshow(im)
        ax.set_axis_off()
        ax.set_title(f"graine {gr} · {NOM_SCORE} = {fmt(s, 3)}", fontsize=12)
    fig.suptitle(f"Même prompt, graines différentes : moyenne {fmt(scores.mean(), 3)} ± {fmt(scores.std(), 3)}",
                 x=0.01, ha="left", fontweight="bold")
    etiqueter(fig, "reel", config_generation["id"])
    plt.show()
    interpretation("Le score varie d'une graine à l'autre : on rapporte une moyenne sur plusieurs images. "
                   "Cinq images restent toutefois très loin des milliers nécessaires à un FID.")

    # Distributions de caractéristiques : images générées et images réelles dans l'espace CLIP (projection 2D par ACP)
    images_reelles = [photo_cafe] + [Image.open(f).convert("RGB") for f in sorted((ASSETS / "recherche" / "images").glob("*.jpg"))]
    vec_gen = encoder_images(CLIP, [image_generee] + images_graines).numpy()
    vec_reel = encoder_images(CLIP, images_reelles).numpy()
    tous = np.vstack([vec_gen, vec_reel])
    centre = tous.mean(0)
    _, _, axes_acp = np.linalg.svd(tous - centre, full_matrices=False)
    p_gen, p_reel = (vec_gen - centre) @ axes_acp[:2].T, (vec_reel - centre) @ axes_acp[:2].T
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(*p_reel[1:].T, s=70, color=PALETTE["gris"], label="photos réelles de la section 5 (autres sujets)")
    ax.scatter(*p_reel[:1].T, s=160, marker="*", color=PALETTE["sarcelle"], label="photo réelle de l'expresso")
    ax.scatter(*p_gen.T, s=80, color=PALETTE["terracotta"], label=f"images générées ({len(vec_gen)})")
    ax.set_xlabel("composante principale 1")
    ax.set_ylabel("composante principale 2")
    ax.legend(loc="best")
    ax.set_title("Caractéristiques CLIP des images générées et d'images réelles (projection 2D par ACP)", loc="left", fontsize=13)
    etiqueter(fig, "reel", "trop peu d'images pour un FID")
    plt.show()
    cos_cafe, cos_autres = float((vec_gen @ vec_reel[0]).mean()), float((vec_gen @ vec_reel[1:].T).mean())
    interpretation(f"Cosinus moyen des images générées avec la photo réelle d'expresso : {fmt(cos_cafe, 3)} ; avec les autres photos : "
                   f"{fmt(cos_autres, 3)}. Le FID compare ce type de nuages, mais avec des milliers d'images réelles et générées "
                   "et les caractéristiques d'Inception : ici, la figure montre l'idée, sans calculer de FID.")
else:
    print("Comparaison sur plusieurs graines : disponible en mode B lorsque la génération a eu lieu.")

### Conclusion
Le **FID** juge un *ensemble* d'images (réalisme, diversité) et ignore le texte ; **CLIPScore** juge chaque image face à *son* texte et hérite des biais de CLIP ; l'**Inception Score** mesure netteté et variété sans comparer aux images réelles. Aucun ne remplace une évaluation humaine.

---
# 3. OCR — diapositives 10 à 12

**Objectif.** Lire la facture de la diapositive 10, comparer le texte lu à la **transcription vérifiée**, puis mesurer CER, WER et correspondance exacte des champs.

**Modèle** : Tesseract 5 (réseau LSTM, modèle français). Programme externe : `apt install tesseract-ocr tesseract-ocr-fra` (Colab, Ubuntu) ou installateur UB Mannheim (Windows, voir README).

In [ ]:
schema_pipeline(["Image", "Prétraitement", "Détection des lignes", "Reconnaissance", "Texte"],
                "Chaîne de traitement de l'OCR (Tesseract 5)",
                ["facture scannée", "binarisation", "blocs, lignes, mots", "LSTM, modèle « fra »", "champs exploitables"],
                mise_en_avant=3)

In [ ]:
import shutil
import pytesseract

if platform.system() == "Windows" and shutil.which("tesseract") is None:
    chemin_windows = Path(r"C:\Program Files\Tesseract-OCR\tesseract.exe")
    if chemin_windows.exists():
        pytesseract.pytesseract.tesseract_cmd = str(chemin_windows)
try:
    VERSION_TESSERACT = str(pytesseract.get_tesseract_version())
    TESSERACT_OK = "fra" in pytesseract.get_languages(config="")
    print(f"Tesseract {VERSION_TESSERACT} détecté · modèle français : {'disponible' if TESSERACT_OK else 'ABSENT'}")
except Exception as erreur:
    TESSERACT_OK = False
    print(f"Tesseract introuvable ({type(erreur).__name__}) : installez-le (voir ci-dessus), puis relancez cette cellule.")

### 3.1 Entrée : la facture, dans deux conditions de numérisation
La facture est **fictive** ; sa transcription de référence est le texte exact qui a servi à fabriquer l'image.

In [ ]:
reference_ocr = json.loads((ASSETS / "ocr" / "facture_reference.json").read_text(encoding="utf-8"))["lignes"]
scan = Image.open(ASSETS / "ocr" / "facture_scan.jpg")
scan_degrade = Image.open(ASSETS / "ocr" / "facture_scan_degradee.jpg")
print(f"Transcription de référence : {len(reference_ocr)} lignes, "
      f"{sum(len(l['texte']) for l in reference_ocr)} caractères, {sum(len(l['texte'].split()) for l in reference_ocr)} mots.")

fig, axes = plt.subplots(1, 2, figsize=(18, 8.4))
for ax, im, titre in [(axes[0], scan, "Scan de la présentation"), (axes[1], scan_degrade, "Condition dégradée")]:
    ax.imshow(im)
    ax.set_axis_off()
    ax.set_title(titre, fontsize=14)
fig.suptitle("Entrée : la facture fictive de la diapositive 10, dans deux conditions de numérisation", x=0.01, ha="left", fontweight="bold")
etiqueter(fig, "donnees")
plt.show()
display(pd.DataFrame([(l["champ"], l["texte"]) for l in reference_ocr], columns=["Ligne (champ)", "Texte de référence"]).head(8))

### 3.2 Prédiction : texte et boîtes détectés

In [ ]:
def lancer_ocr(image, psm=4):
    donnees = pytesseract.image_to_data(image, lang="fra", config=f"--oem 1 --psm {psm}",
                                        output_type=pytesseract.Output.DATAFRAME)
    mots = donnees[(donnees.level == 5) & donnees.text.notna()].copy()  # niveau 5 = mot
    mots["text"] = mots.text.astype(str).str.strip()
    return mots[mots.text != ""].reset_index(drop=True)


if TESSERACT_OK:
    with chrono("OCR du scan"):
        mots_scan = lancer_ocr(scan)
    print(f"{len(mots_scan)} mots reconnus · confiance médiane : {mots_scan.conf.median():.0f} / 100")
    display(mots_scan[["left", "top", "width", "height", "conf", "text"]].head(8))
else:
    mots_scan = None
    print("OCR non exécuté : Tesseract n'est pas disponible.")

In [ ]:
def regrouper_par_ligne(mots, reference, marge=8):
    # Rattache chaque mot reconnu à la ligne de référence qui contient son centre (la plus proche verticalement)
    textes, hors_reference = {i: [] for i in range(len(reference))}, []
    for _, m in mots.iterrows():
        cx, cy = m.left + m.width / 2, m.top + m.height / 2
        candidats = [i for i, l in enumerate(reference)
                     if l["boite"][0] - marge <= cx <= l["boite"][2] + marge and l["boite"][1] - marge <= cy <= l["boite"][3] + marge]
        if candidats:
            i = min(candidats, key=lambda k: abs(cy - (reference[k]["boite"][1] + reference[k]["boite"][3]) / 2))
            textes[i].append((m.left, m.text))
        else:
            hors_reference.append((m.left, m.text))
    return ({i: " ".join(t for _, t in sorted(v)) for i, v in textes.items()},
            " ".join(t for _, t in sorted(hors_reference)))


if mots_scan is not None:
    lignes_scan, hors_scan = regrouper_par_ligne(mots_scan, reference_ocr)
    fig, axes = plt.subplots(1, 2, figsize=(19, 8.6), gridspec_kw={"width_ratios": [1.05, 1]})
    axes[0].imshow(scan)
    axes[0].set_axis_off()
    for _, m in mots_scan.iterrows():
        c = PALETTE["sauge"] if m.conf >= 80 else (PALETTE["ocre"] if m.conf >= 50 else PALETTE["terracotta"])
        axes[0].add_patch(patches.Rectangle((m.left, m.top), m.width, m.height, fill=False, ec=c, lw=1.6))
    axes[0].legend(handles=[patches.Patch(fc="none", ec=PALETTE["sauge"], label="confiance ≥ 80"),
                            patches.Patch(fc="none", ec=PALETTE["ocre"], label="50 à 80"),
                            patches.Patch(fc="none", ec=PALETTE["terracotta"], label="< 50")], loc="lower right", fontsize=11)
    axes[0].set_title("Mots détectés par Tesseract", fontsize=14)
    axes[1].set_axis_off()
    texte_ocr = "\n".join(f"{(lignes_scan[i] or '∅')}" for i in range(len(reference_ocr)))
    axes[1].text(0, 1, texte_ocr, va="top", ha="left", family="monospace", fontsize=11.2, transform=axes[1].transAxes)
    axes[1].set_title("Texte reconnu, rangé par ligne de référence (∅ = rien lu)", fontsize=14, loc="left")
    etiqueter(fig, "reel", f"Tesseract {VERSION_TESSERACT}, modèle fra")
    sauvegarder(fig, "ocr_resultat.png")
    plt.show()

### 3.3 Évaluation

$$\text{CER} = \frac{S + D + I}{N} \qquad \text{WER} = \frac{S_w + D_w + I_w}{N_w} \qquad \text{EM} = \frac{\text{champs exacts}}{\text{champs évalués}}$$

S, D, I : substitutions, suppressions, insertions (distance de Levenshtein) ; N : caractères de la référence. CER et WER : **plus bas = meilleur** ; EM (correspondance exacte) : **plus haut = meilleur**.
Normalisation annoncée : **stricte** (seuls les espaces sont unifiés) ou **souple** (minuscules, sans accents ni ponctuation).

In [ ]:
def normaliser(texte, mode="stricte"):
    t = unicodedata.normalize("NFC", texte).replace("\u00a0", " ").replace("\u202f", " ").replace("\t", " ")
    if mode == "souple":
        t = t.lower()
        t = "".join(c for c in unicodedata.normalize("NFD", t) if unicodedata.category(c) != "Mn")  # sans accents
        t = re.sub(r"[–—‐‑−]", "-", t)
        t = re.sub(r"[’‘`]", "'", t)
        t = re.sub(r"[^\w\s]", " ", t)  # ponctuation supprimée
    return re.sub(r"\s+", " ", t).strip()


for exemple in ["12, rue des Orangers – 10000 Rabat", "Échéance : 13/11/2026"]:
    print(f"{exemple!r:42} stricte → {normaliser(exemple)!r:40} souple → {normaliser(exemple, 'souple')!r}")

In [ ]:
def aligner(reference, hypothese):
    # Programmation dynamique : D[i, j] = distance entre les i premiers éléments de la référence et les j premiers de l'hypothèse
    n, m = len(reference), len(hypothese)
    D = np.zeros((n + 1, m + 1), dtype=int)
    D[:, 0], D[0, :] = np.arange(n + 1), np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = min(D[i - 1, j] + 1, D[i, j - 1] + 1, D[i - 1, j - 1] + (reference[i - 1] != hypothese[j - 1]))
    operations, i, j = [], n, m  # remontée du chemin optimal
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i - 1, j - 1] + (reference[i - 1] != hypothese[j - 1]):
            operations.append(("=" if reference[i - 1] == hypothese[j - 1] else "S", reference[i - 1], hypothese[j - 1]))
            i, j = i - 1, j - 1
        elif i > 0 and D[i, j] == D[i - 1, j] + 1:
            operations.append(("D", reference[i - 1], ""))
            i -= 1
        else:
            operations.append(("I", "", hypothese[j - 1]))
            j -= 1
    return operations[::-1]


def compter_erreurs(operations):
    c = Counter(op for op, _, _ in operations)
    return c["S"], c["D"], c["I"]


import jiwer

ref_diapo, ocr_diapo = "Total TTC : 1 248,00 DH", "Total TTC : 7 248,00 DH"
S, D_, I = compter_erreurs(aligner(ref_diapo, ocr_diapo))
print(f"Exemple de la diapositive : S = {S}, D = {D_}, I = {I}, N = {len(ref_diapo)} → CER = {fmt((S + D_ + I) / len(ref_diapo), 4)}")
print(f"Contrôle avec jiwer       : CER = {fmt(jiwer.cer(ref_diapo, ocr_diapo), 4)} · WER = {fmt(jiwer.wer(ref_diapo, ocr_diapo), 4)}")

In [ ]:
CHAMPS_CRITIQUES = ["Numéro de facture", "Date", "Échéance", "Total HT", "TVA 20 %", "Total TTC"]


def evaluer_ocr(lignes_lues, hors_reference, reference, mode="stricte"):
    lignes, E, N, Ew, Nw = [], 0, 0, 0, 0
    for i, l in enumerate(reference):
        r, h = normaliser(l["texte"], mode), normaliser(lignes_lues[i], mode)
        s, d, ins = compter_erreurs(aligner(r, h))
        sw, dw, iw = compter_erreurs(aligner(r.split(), h.split()))
        lignes.append({"champ": l["champ"], "référence": r, "OCR": h, "S": s, "D": d, "I": ins, "N": len(r),
                       "CER": (s + d + ins) / max(len(r), 1), "exact": r == h})
        E, N, Ew, Nw = E + s + d + ins, N + len(r), Ew + sw + dw + iw, Nw + len(r.split())
    extra = normaliser(hors_reference, mode)  # texte lu hors de toute ligne de référence : ce sont des insertions
    E, Ew = E + len(extra), Ew + len(extra.split())
    detail = pd.DataFrame(lignes)
    critiques = detail[detail.champ.isin(CHAMPS_CRITIQUES)]
    return {"CER": E / N, "WER": Ew / Nw, "exactitude_champs": detail.exact.mean(),
            "exactitude_critiques": critiques.exact.mean(), "S": detail.S.sum(), "D": detail.D.sum(),
            "I": detail.I.sum() + len(extra), "N": N}, detail


if mots_scan is not None:
    bilan_scan = {m: evaluer_ocr(lignes_scan, hors_scan, reference_ocr, m) for m in ["stricte", "souple"]}
    resume = pd.DataFrame({m: b[0] for m, b in bilan_scan.items()}).T.rename(
        columns={"exactitude_champs": "EM (toutes les lignes)", "exactitude_critiques": "EM (champs critiques)"})
    display(resume.style.format({"CER": "{:.2%}", "WER": "{:.2%}", "EM (toutes les lignes)": "{:.0%}",
                                 "EM (champs critiques)": "{:.0%}", "S": "{:.0f}", "D": "{:.0f}", "I": "{:.0f}", "N": "{:.0f}"}))
    detail_scan = bilan_scan["stricte"][1]
    # Contrôle croisé du CER strict avec jiwer (somme des modifications / somme des longueurs de référence)
    refs = [r for r in detail_scan["référence"]]
    hyps = [h for h in detail_scan["OCR"]]
    print(f"Contrôle jiwer (lignes) : CER = {jiwer.cer(refs, hyps):.4%} · notre calcul : "
          f"{(detail_scan.S.sum() + detail_scan.D.sum() + detail_scan.I.sum()) / detail_scan.N.sum():.4%}")
    RESULTATS["ocr"] = {"CER": bilan_scan["stricte"][0]["CER"], "WER": bilan_scan["stricte"][0]["WER"],
                        "CER_souple": bilan_scan["souple"][0]["CER"],
                        "exactitude_critiques": bilan_scan["stricte"][0]["exactitude_critiques"]}

In [ ]:
def surligner(reference, hypothese):
    # Affichage HTML des différences caractère par caractère (␣ = espace)
    def bloc(c, fond):
        c = "␣" if c == " " else (c.replace("&", "&amp;").replace("<", "&lt;") or "∅")
        return f'<span style="background:{fond};color:#19334A;border-radius:3px;padding:0 1px">{c}</span>'
    ligne_r, ligne_h = [], []
    for op, r, h in aligner(reference, hypothese):
        if op == "=":
            ligne_r.append(r.replace("&", "&amp;").replace("<", "&lt;"))
            ligne_h.append(h.replace("&", "&amp;").replace("<", "&lt;"))
        elif op == "S":
            ligne_r.append(bloc(r, "#EBD9A8"))
            ligne_h.append(bloc(h, "#EBD9A8"))
        elif op == "D":
            ligne_r.append(bloc(r, "#F0C9BC"))
            ligne_h.append(bloc("", "#F0C9BC"))
        else:
            ligne_h.append(bloc(h, "#F0C9BC"))
    return "".join(ligne_r), "".join(ligne_h)


def afficher_differences(detail):
    lignes_html = []
    for _, l in detail[~detail.exact].iterrows():
        r, h = surligner(l["référence"], l["OCR"])
        lignes_html.append(f"<tr><td style='padding:4px 10px;color:#5F6B78'>{l['champ']}</td>"
                           f"<td style='font-family:monospace;padding:4px 10px'>réf. : {r}<br>OCR : {h}</td>"
                           f"<td style='padding:4px 10px'>S={l.S} · D={l.D} · I={l.I}</td></tr>")
    legende = ("<p style='font-size:13px'><span style='background:#EBD9A8'>&nbsp;substitution&nbsp;</span> &nbsp; "
               "<span style='background:#F0C9BC'>&nbsp;suppression ou insertion&nbsp;</span> &nbsp; ␣ = espace · ∅ = caractère absent</p>")
    display(HTML(legende + "<table style='font-size:14px;border-collapse:collapse'>" + "".join(lignes_html) + "</table>"
                 if lignes_html else "<p>Aucune différence : toutes les lignes sont exactes.</p>"))


if mots_scan is not None:
    afficher_differences(detail_scan)
    b = bilan_scan["stricte"][0]
    interpretation(f"CER strict = {fmt(b['CER'], pct=True)} et WER = {fmt(b['WER'], pct=True)} ; en normalisation souple, "
                   f"le CER tombe à {fmt(bilan_scan['souple'][0]['CER'], pct=True)}. Une partie des « erreurs » est donc typographique "
                   "(tiret long ou court, espace) : c'est pourquoi la normalisation doit toujours être annoncée. "
                   f"Champs critiques exacts : {fmt(b['exactitude_critiques'], pct=True)}.")

### 3.4 Même document, condition dégradée

In [ ]:
if TESSERACT_OK:
    with chrono("OCR du scan dégradé"):
        mots_degrade = lancer_ocr(scan_degrade)
    lignes_degrade, hors_degrade = regrouper_par_ligne(mots_degrade, reference_ocr)
    bilan_degrade = {m: evaluer_ocr(lignes_degrade, hors_degrade, reference_ocr, m) for m in ["stricte", "souple"]}
    detail_degrade = bilan_degrade["stricte"][1]

    conditions = {"scan de la présentation": bilan_scan, "condition dégradée": bilan_degrade}
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.6))
    for ax, (cle, titre, sens) in zip(axes, [("CER", "CER (plus bas = meilleur)", "↓"), ("WER", "WER (plus bas = meilleur)", "↓"),
                                             ("exactitude_critiques", "EM des champs critiques (plus haut = meilleur)", "↑")]):
        for k, (nom, bilan) in enumerate(conditions.items()):
            for j, mode in enumerate(["stricte", "souple"]):
                v = bilan[mode][0][cle]
                x = k * 2.6 + j
                ax.bar(x, 100 * v, color=[PALETTE["sarcelle"], PALETTE["sauge"]][j], width=0.9)
                ax.text(x, 100 * v + 1, fmt(v, pct=True), ha="center", fontsize=11)
        ax.set_xticks([0.5, 3.1], list(conditions))
        ax.set_ylabel("%")
        ax.set_ylim(0, 1.25 * max(100 * b[m][0][cle] for b in conditions.values() for m in ["stricte", "souple"]) + 1)
        ax.set_title(titre, fontsize=13)
    axes[0].legend(handles=[patches.Patch(color=PALETTE["sarcelle"], label="normalisation stricte"),
                            patches.Patch(color=PALETTE["sauge"], label="normalisation souple")], loc="upper left")
    fig.suptitle("OCR réel : effet de la qualité de numérisation et de la normalisation", x=0.01, ha="left", fontweight="bold")
    fig.subplots_adjust(top=0.8, wspace=0.3)
    etiqueter(fig, "reel", f"Tesseract {VERSION_TESSERACT}")
    sauvegarder(fig, "ocr_conditions.png")
    plt.show()
    erreurs_critiques = detail_degrade[detail_degrade.champ.isin(CHAMPS_CRITIQUES) & ~detail_degrade.exact]
    display(Markdown("**Champs critiques mal lus dans la condition dégradée :**"))
    if len(erreurs_critiques):
        afficher_differences(erreurs_critiques)
    else:
        print("aucun")
    RESULTATS["ocr"]["CER_degrade"] = bilan_degrade["stricte"][0]["CER"]
    interpretation(f"Le CER strict passe de {fmt(bilan_scan['stricte'][0]['CER'], pct=True)} à "
                   f"{fmt(bilan_degrade['stricte'][0]['CER'], pct=True)}. Les erreurs touchent aussi des **montants** : "
                   "un séparateur décimal perdu ou un chiffre mal lu rend le champ inutilisable, même si le CER global reste modéré.")

In [ ]:
if TESSERACT_OK:
    fig, ax = plt.subplots(figsize=(13, 3.2))
    for k, (nom, bilan) in enumerate([("scan de la présentation", bilan_scan), ("condition dégradée", bilan_degrade)]):
        b, gauche = bilan["stricte"][0], 0
        for cle, couleur, libelle in [("S", PALETTE["ocre"], "substitutions"), ("D", PALETTE["terracotta"], "suppressions"),
                                      ("I", PALETTE["marine"], "insertions")]:
            ax.barh(k, b[cle], left=gauche, color=couleur, label=libelle if k == 0 else None)
            if b[cle] >= 8:
                ax.text(gauche + b[cle] / 2, k, str(int(b[cle])), ha="center", va="center", color="white", fontweight="bold")
            gauche += b[cle]
        ax.text(gauche + 3, k, f"S + D + I = {int(gauche)} sur N = {int(b['N'])} → CER = {fmt(b['CER'], pct=True)}", va="center", fontsize=12)
    ax.set_yticks([0, 1], ["scan de la présentation", "condition dégradée"])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.6 * (bilan_degrade["stricte"][0]["S"] + bilan_degrade["stricte"][0]["D"] + bilan_degrade["stricte"][0]["I"]))
    ax.set_xlabel("Nombre d'opérations d'édition (normalisation stricte)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=3, frameon=False)
    ax.set_title("Nature des erreurs : de quoi est fait le CER ?", loc="left")
    etiqueter(fig, "reel", f"Tesseract {VERSION_TESSERACT}")
    plt.show()


    def explorer_ocr(condition, normalisation, champ):
        bilan = (bilan_scan if condition == "scan de la présentation" else bilan_degrade)[normalisation]
        ligne = bilan[1][bilan[1].champ == champ].iloc[0]
        r, h = surligner(ligne["référence"], ligne["OCR"])
        display(HTML(f"<p style='font-size:15px'><b>{champ}</b> · S = {ligne.S}, D = {ligne.D}, I = {ligne.I}, N = {ligne.N} → "
                     f"CER de la ligne = {fmt(ligne.CER, pct=True)} · {'exact' if ligne.exact else 'non exact'}</p>"
                     f"<p style='font-family:monospace;font-size:16px'>réf. : {r}<br>OCR : {h}</p>"
                     f"<p>Document entier : CER = {fmt(bilan[0]['CER'], pct=True)} · WER = {fmt(bilan[0]['WER'], pct=True)} · "
                     f"EM des champs critiques = {fmt(bilan[0]['exactitude_critiques'], pct=True)}</p>"))

    if WIDGETS_OK:
        explorer(explorer_ocr,
                 condition=widgets.Dropdown(options=["scan de la présentation", "condition dégradée"], value="condition dégradée", description="document"),
                 normalisation=widgets.Dropdown(options=["stricte", "souple"], description="normalisation"),
                 champ=widgets.Dropdown(options=[l["champ"] for l in reference_ocr], value="Total TTC", description="ligne"))
    else:
        explorer_ocr("condition dégradée", "stricte", "Total TTC")

### 3.5 Calcul de la diapositive 12 : une seule erreur de chiffre
Erreur **construite à la main** (sur le vrai scan, Tesseract lit correctement le total).

In [ ]:
montant_vrai, montant_lu = 1248.00, 7248.00
ops = aligner(ref_diapo, ocr_diapo)
S, D_, I = compter_erreurs(ops)
print(f"Référence : {ref_diapo}\nLecture   : {ocr_diapo}")
print(f"CER = (S + D + I) / N = ({S} + {D_} + {I}) / {len(ref_diapo)} = {fmt((S + D_ + I) / len(ref_diapo), pct=True)}")
print(f"WER = 1 mot faux sur {len(ref_diapo.split())} = {fmt(1 / len(ref_diapo.split()), pct=True)}")
print(f"Écart sur le montant : {montant_lu - montant_vrai:,.0f} DH".replace(",", " "))
r, h = surligner(ref_diapo, ocr_diapo)
display(HTML(f"<p style='font-family:monospace;font-size:17px'>réf. : {r}<br>OCR : {h}</p>"))
interpretation("4,3 % d'erreur sur les caractères, mais 6 000 DH d'écart : le CER traite tous les caractères de la même façon. "
               "Pour un usage métier, on mesure aussi la correspondance exacte des champs critiques.")

### Conclusion
Le CER compte les modifications par caractère et dépend de la **normalisation** annoncée ; le WER est presque toujours plus élevé (un caractère faux rend tout le mot faux). Aucun des deux ne sait qu'un chiffre de montant compte plus qu'un espace : la **correspondance exacte** des champs critiques complète l'évaluation.

---
# 4. Légendage d'images — diapositives 13 à 15

**Objectif.** Décrire la photo de la diapositive 13 avec BLIP, comparer la phrase aux **cinq légendes humaines** de Flickr8k, et montrer ce que BLEU, METEOR et CIDEr mesurent… et ce qu'ils ne mesurent pas.

**Modèle** : BLIP base (`Salesforce/blip-image-captioning-base`, ≈ 1 Go, en anglais ; version *large* en mode B).

In [ ]:
schema_pipeline(["Image", "Encodeur visuel", "Décodeur de texte", "Légende"], "Chaîne de légendage (BLIP)",
                ["photo 384 × 384", "Vision Transformer", "mot après mot", "phrase en anglais"], mise_en_avant=2)

### 4.1 Entrée : la photo et ses références humaines

In [ ]:
donnees_legende = json.loads((ASSETS / "legendage" / "references.json").read_text(encoding="utf-8"))
photo_quai = Image.open(ASSETS / "legendage" / donnees_legende["image"]).convert("RGB")
REFERENCES_EN = donnees_legende["references_anglais_flickr8k"]
EXEMPLE_FR = donnees_legende["exemple_pedagogique_francais"]

fig, axes = plt.subplots(1, 2, figsize=(17, 4.8), gridspec_kw={"width_ratios": [1, 1.25]})
axes[0].imshow(photo_quai)
axes[0].set_axis_off()
axes[0].set_title("Flickr8k · 2414384480_096867d695.jpg", fontsize=13)
axes[1].set_axis_off()
axes[1].text(0, 0.95, "Cinq légendes humaines (anglais, Flickr8k) :", fontsize=14, fontweight="bold", transform=axes[1].transAxes, va="top")
for k, r in enumerate(REFERENCES_EN):
    axes[1].text(0, 0.8 - 0.14 * k, f"{k + 1}. {r}", fontsize=13, transform=axes[1].transAxes, va="top")
etiqueter(fig, "donnees")
plt.show()

### 4.2 Prédiction : légende réelle de BLIP
En **sauge**, les mots présents dans au moins une référence ; en **terracotta**, les mots absents (pas forcément faux).

In [ ]:
def legender(nom_modele, image):
    from transformers import BlipForConditionalGeneration, BlipProcessor
    processeur = BlipProcessor.from_pretrained(nom_modele)
    modele = BlipForConditionalGeneration.from_pretrained(nom_modele).to(DEVICE).eval()
    entrees = processeur(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        sortie = modele.generate(**entrees, max_new_tokens=30, num_beams=3)
    return processeur.decode(sortie[0], skip_special_tokens=True).strip()


LEGENDES_MODELES = {}
modeles_blip = ["Salesforce/blip-image-captioning-base"] + (["Salesforce/blip-image-captioning-large"] if MODE == "B" else [])
for nom in modeles_blip:
    try:
        with chrono(f"Légendage avec {nom}"):
            LEGENDES_MODELES[nom] = legender(nom, photo_quai)
        print(f"  → « {LEGENDES_MODELES[nom]} »")
    except Exception as erreur:
        print(f"{nom} indisponible ({type(erreur).__name__}: {str(erreur)[:120]}). Aucune légende n'est inventée à sa place.")
LEGENDE_BLIP = LEGENDES_MODELES.get("Salesforce/blip-image-captioning-base")

In [ ]:
def tokeniser(texte):
    # Convention de comparaison : minuscules, ponctuation retirée
    return re.findall(r"\w+", texte.lower())


def ecrire_mots_colores(ax, x, y, mots, couleurs, taille=15):
    # Écrit une phrase mot par mot, chaque mot avec sa couleur (coordonnées relatives à l'axe)
    rendu = ax.figure.canvas.get_renderer()
    for mot, couleur in zip(mots, couleurs):
        t = ax.text(x, y, mot + " ", color=couleur, fontsize=taille, fontweight="bold", transform=ax.transAxes, va="top")
        x = t.get_window_extent(renderer=rendu).transformed(ax.transAxes.inverted()).x1


if LEGENDE_BLIP:
    vocabulaire_refs = set(w for r in REFERENCES_EN for w in tokeniser(r))
    mots = tokeniser(LEGENDE_BLIP)
    fig, axes = plt.subplots(1, 2, figsize=(17, 4.8), gridspec_kw={"width_ratios": [1, 1.25]})
    axes[0].imshow(photo_quai)
    axes[0].set_axis_off()
    axes[1].set_axis_off()
    axes[1].text(0, 0.95, "Légende générée par BLIP :", fontsize=14, transform=axes[1].transAxes, va="top")
    ecrire_mots_colores(axes[1], 0, 0.82, mots, [PALETTE["sauge"] if w in vocabulaire_refs else PALETTE["terracotta"] for w in mots], 17)
    axes[1].text(0, 0.6, "Référence la plus proche :", fontsize=14, transform=axes[1].transAxes, va="top")
    proche = max(REFERENCES_EN, key=lambda r: len(set(tokeniser(r)) & set(mots)))
    axes[1].text(0, 0.48, proche, fontsize=14, transform=axes[1].transAxes, va="top", color=PALETTE["marine"])
    axes[1].legend(handles=[patches.Patch(color=PALETTE["sauge"], label="mot présent dans une référence"),
                            patches.Patch(color=PALETTE["terracotta"], label="mot absent des références")],
                   loc="lower left", fontsize=12)
    etiqueter(fig, "reel", "BLIP base")
    sauvegarder(fig, "legendage_blip.png")
    plt.show()
else:
    print("Pas de légende générée dans cette exécution : l'exemple de la diapositive 15 et la comparaison 4.4 restent calculés.")

### 4.3 Évaluation : BLEU-1 (diapositive 15)
$\text{BLEU-1} = \text{BP} \times p_1$, avec $p_1$ = mots communs (écrêtés) / mots générés et $\text{BP} = 1$ si $c > r$, sinon $e^{1 - r/c}$. **Plus haut = meilleur.** Exemple de la diapositive, en français, sans traduire de sortie de modèle :

In [ ]:
def bleu1_detaille(candidate, references):
    # BLEU-1 « à la main » : précision écrêtée des mots + pénalité de brièveté (référence la plus proche en longueur)
    c, refs = tokeniser(candidate), [tokeniser(r) for r in references]
    max_ref = Counter()
    for r in refs:
        for mot, k in Counter(r).items():
            max_ref[mot] = max(max_ref[mot], k)
    apparies = {mot: min(k, max_ref[mot]) for mot, k in Counter(c).items()}
    p1 = sum(apparies.values()) / len(c)
    r_len = min((abs(len(r) - len(c)), len(r)) for r in refs)[1]
    bp = 1.0 if len(c) > r_len else math.exp(1 - r_len / len(c))
    return {"p1": p1, "BP": bp, "BLEU-1": bp * p1, "apparies": sum(apparies.values()), "c": len(c), "r": r_len}


from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu

ref_fr = EXEMPLE_FR["reference"]
for nom, cand in [("légende correcte", EXEMPLE_FR["candidate_correcte"]), ("légende fausse (« bus »)", EXEMPLE_FR["candidate_fausse"])]:
    d = bleu1_detaille(cand, [ref_fr])
    nltk_b1 = sentence_bleu([tokeniser(ref_fr)], tokeniser(cand), weights=(1, 0, 0, 0))
    print(f"{nom:26} « {cand} »")
    print(f"{'':26} p1 = {d['apparies']}/{d['c']} = {fmt(d['p1'], 3)} ; c = {d['c']}, r = {d['r']} → BP = {fmt(d['BP'])} ; "
          f"BLEU-1 = {fmt(d['BLEU-1'], 3)} (NLTK : {fmt(nltk_b1, 3)})")

mots_ref = set(tokeniser(ref_fr))
fig, axes = plt.subplots(1, 2, figsize=(18, 4.4), gridspec_kw={"width_ratios": [1.5, 1]})
ax = axes[0]
ax.set_axis_off()
ax.text(0, 0.95, "Référence :", fontsize=13, color=PALETTE["gris"], transform=ax.transAxes, va="top")
ax.text(0.17, 0.95, ref_fr, fontsize=15, transform=ax.transAxes, va="top")
for k, (nom, cand) in enumerate([("Correcte :", EXEMPLE_FR["candidate_correcte"]), ("Fausse :", EXEMPLE_FR["candidate_fausse"])]):
    y = 0.62 - 0.33 * k
    ax.text(0, y, nom, fontsize=13, color=PALETTE["gris"], transform=ax.transAxes, va="top")
    m = tokeniser(cand)
    ecrire_mots_colores(ax, 0.17, y, m, [PALETTE["sauge"] if w in mots_ref else PALETTE["terracotta"] for w in m], 15)
ax = axes[1]
valeurs = [bleu1_detaille(EXEMPLE_FR["candidate_correcte"], [ref_fr])["BLEU-1"], bleu1_detaille(EXEMPLE_FR["candidate_fausse"], [ref_fr])["BLEU-1"]]
ax.barh(["correcte\n(voyageurs, train)", "fausse\n(bus)"], valeurs, color=[PALETTE["sauge"], PALETTE["terracotta"]])
for k, v in enumerate(valeurs):
    ax.text(v + 0.01, k, f"{fmt(v, 3)}", va="center", fontsize=14, fontweight="bold")
ax.set_xlim(0, 1.05)
ax.invert_yaxis()
ax.set_xlabel("BLEU-1")
ax.set_title("La légende fausse obtient le meilleur score", fontsize=13, loc="left")
etiqueter(fig, "pedagogique", "diapositive 15")
plt.show()
interpretation("« voyageurs » et « train » sont corrects mais absents de la référence : ils comptent comme des erreurs. "
               "« bus » est faux, mais la phrase partage 7 mots sur 8 avec la référence. BLEU mesure un **recouvrement lexical**, pas le sens.")

**Mesure réelle** : légende de BLIP contre les cinq références anglaises (même langue, même tokenisation : minuscules, sans ponctuation). METEOR apparie aussi radicaux et synonymes (WordNet, anglais).

In [ ]:
import nltk
from nltk.translate.meteor_score import meteor_score

for ressource in ["wordnet", "omw-1.4"]:
    try:
        nltk.download(ressource, quiet=True)
    except Exception:
        pass
try:
    from nltk.corpus import wordnet
    wordnet.synsets("train")  # déclenche le chargement : lève LookupError si la ressource manque
    METEOR_OK = True
except LookupError:
    METEOR_OK = False
    print("Ressource WordNet indisponible (téléchargement impossible) : METEOR ne sera pas calculé et apparaîtra comme « — ».")
LISSAGE = SmoothingFunction().method1


def scores_legende(candidate, references):
    c, refs = tokeniser(candidate), [tokeniser(r) for r in references]
    d = bleu1_detaille(candidate, references)
    resultat = {"p1": d["p1"], "BP": d["BP"], "BLEU-1": d["BLEU-1"],
                "BLEU-1 (NLTK)": sentence_bleu(refs, c, weights=(1, 0, 0, 0)),
                "BLEU-4 sans lissage": sentence_bleu(refs, c, weights=(0.25,) * 4),
                "BLEU-4 lissé": sentence_bleu(refs, c, weights=(0.25,) * 4, smoothing_function=LISSAGE)}
    try:
        resultat["METEOR"] = meteor_score(refs, c)
    except LookupError:
        resultat["METEOR"] = float("nan")  # ressource WordNet absente (téléchargement impossible)
    return resultat


if LEGENDE_BLIP:
    tableau_blip = pd.DataFrame({nom.split("/")[-1]: scores_legende(leg, REFERENCES_EN) for nom, leg in LEGENDES_MODELES.items()}).T
    display(tableau_blip.style.format("{:.3f}", na_rep="—"))
    s = tableau_blip.iloc[0]
    RESULTATS["legendage"] = {"BLEU-1": s["BLEU-1"], "BLEU-4 lissé": s["BLEU-4 lissé"], "METEOR": s["METEOR"], "legende": LEGENDE_BLIP}
    interpretation(f"BLEU-1 = {fmt(s['BLEU-1'], 3)} (vérifié par NLTK) ; BLEU-4 sans lissage = {fmt(s['BLEU-4 sans lissage'], 3)}, "
                   f"lissé = {fmt(s['BLEU-4 lissé'], 3)} ; METEOR = {fmt(s['METEOR'], 3)}. Sur une seule image, ces valeurs illustrent "
                   "le calcul : elles ne mesurent pas la qualité générale du modèle, qui s'évalue sur des milliers d'images.")
else:
    print("Mesure réelle non disponible : BLIP n'a pas pu être exécuté dans cet environnement.")

### 4.4 Plusieurs candidates, trois métriques : BLEU, METEOR et CIDEr
**CIDEr-D** (Vedantam et al., 2015) : cosinus entre vecteurs TF-IDF de n-grammes (n = 1 à 4), moyenné sur les références, avec pénalité de longueur, × 10. L'IDF est calculé sur les références des **12 images** de la collection (section 5) : un n-gramme présent partout (« a man ») pèse peu.

Pour une comparaison équitable, la légende humaine n° 1 est **retirée des références** et jouée comme candidate : c'est un repère « niveau humain ». Les autres candidates : la légende réelle de BLIP (si disponible), une paraphrase fidèle et une phrase fausse (exemples écrits à la main).

In [ ]:
def ngrammes(mots, n_max=4):
    return Counter(tuple(mots[i:i + n]) for n in range(1, n_max + 1) for i in range(len(mots) - n + 1))


class CIDErD:
    # Réimplémentation de CIDEr-D (pycocoevalcap) : TF-IDF des n-grammes, écrêtage, pénalité gaussienne de longueur (σ = 6), × 10
    def __init__(self, corpus_references, sigma=6.0):
        self.df = Counter(g for refs in corpus_references for g in {g for r in refs for g in ngrammes(r)})
        self.log_n = np.log(len(corpus_references))
        self.sigma = sigma

    def _vecteur(self, mots):
        vec, norme = [dict() for _ in range(4)], np.zeros(4)
        for g, tf in ngrammes(mots).items():
            vec[len(g) - 1][g] = tf * (self.log_n - np.log(max(1.0, self.df[g])))
            norme[len(g) - 1] += vec[len(g) - 1][g] ** 2
        return vec, np.sqrt(norme), max(len(mots) - 1, 0)  # longueur comptée en bigrammes, comme pycocoevalcap

    def score(self, candidate, references):
        vc, nc, lc = self._vecteur(candidate)
        total = np.zeros(4)
        for r in references:
            vr, nr, lr = self._vecteur(r)
            for n in range(4):
                val = sum(min(v, vr[n].get(g, 0.0)) * vr[n].get(g, 0.0) for g, v in vc[n].items())
                if nc[n] and nr[n]:
                    val /= nc[n] * nr[n]
                total[n] += val * np.exp(-((lc - lr) ** 2) / (2 * self.sigma ** 2))
        return float(total.mean() / len(references) * 10)


collection_legendes = json.loads((ASSETS / "recherche" / "collection.json").read_text(encoding="utf-8"))["images"]
REFS_EVAL = REFERENCES_EN[1:]  # la référence n° 1 devient une candidate : elle ne peut pas être sa propre référence
corpus = [[tokeniser(r) for r in (REFS_EVAL if im["id"].startswith("2414384480") else im["legendes_flickr8k"])]
          for im in collection_legendes]
cider = CIDErD(corpus)

candidates_4 = {"humaine n° 1 (retirée des références)": REFERENCES_EN[0]}
if LEGENDE_BLIP:
    candidates_4["BLIP (réelle)"] = LEGENDE_BLIP
candidates_4["paraphrase fidèle (écrite à la main)"] = "commuters stand by the tracks as a metro arrives"
candidates_4["phrase fausse (« bus », écrite à la main)"] = "a group of people are standing on a platform waiting for the bus"


def tous_les_scores(candidate):
    s = scores_legende(candidate, REFS_EVAL)
    return {"BLEU-1": s["BLEU-1"], "BLEU-4 lissé": s["BLEU-4 lissé"], "METEOR": s["METEOR"],
            "CIDEr-D": cider.score(tokeniser(candidate), [tokeniser(r) for r in REFS_EVAL])}


tableau_lex = pd.DataFrame({nom: tous_les_scores(c) for nom, c in candidates_4.items()}).T
display(pd.concat([pd.Series(candidates_4, name="phrase"), tableau_lex], axis=1).style.format("{:.3f}", subset=list(tableau_lex.columns), na_rep="—"))

couleurs = {"humaine n° 1 (retirée des références)": PALETTE["marine"], "BLIP (réelle)": PALETTE["sarcelle"],
            "paraphrase fidèle (écrite à la main)": PALETTE["sauge"], "phrase fausse (« bus », écrite à la main)": PALETTE["terracotta"]}
fig, axes = plt.subplots(1, 2, figsize=(17, 4.8), gridspec_kw={"width_ratios": [3, 1.3]})
metriques = ["BLEU-1", "BLEU-4 lissé", "METEOR"]
x = np.arange(len(metriques))
largeur = 0.8 / len(tableau_lex)
for k, (nom, ligne) in enumerate(tableau_lex.iterrows()):
    axes[0].bar(x + k * largeur, ligne[metriques].values.astype(float), width=largeur, color=couleurs[nom], label=nom)
    axes[1].bar(k, ligne["CIDEr-D"], color=couleurs[nom])
    axes[1].text(k, ligne["CIDEr-D"] + 0.03, fmt(ligne["CIDEr-D"], 2), ha="center", fontsize=11)
    for xi, v in zip(x + k * largeur, ligne[metriques].values.astype(float)):
        if not np.isnan(v):
            axes[0].text(xi, v + 0.015, fmt(v, 2), ha="center", fontsize=10)
axes[0].set_xticks(x + largeur * (len(tableau_lex) - 1) / 2, metriques)
axes[0].set_ylim(0, 1.15)
axes[0].legend(loc="upper right", fontsize=10.5)
axes[0].set_title("BLEU et METEOR (entre 0 et 1)", loc="left", fontsize=13)
axes[1].set_xticks([])
axes[1].set_title("CIDEr-D (échelle × 10)", loc="left", fontsize=13)
axes[1].set_ylim(0, 1.25 * np.nanmax(tableau_lex["CIDEr-D"].values.astype(float)))
fig.suptitle("Même photo, plusieurs légendes candidates, quatre références humaines", x=0.01, ha="left", fontweight="bold")
fig.subplots_adjust(top=0.84)
etiqueter(fig, "reel" if LEGENDE_BLIP else "pedagogique", "BLIP réel" if LEGENDE_BLIP else "phrases écrites à la main")
sauvegarder(fig, "legendage_metriques.png")
plt.show()
fausse, fidele = tableau_lex.loc["phrase fausse (« bus », écrite à la main)"], tableau_lex.loc["paraphrase fidèle (écrite à la main)"]
devance = [m for m in tableau_lex.columns if not np.isnan(fausse[m]) and fausse[m] > fidele[m]]
RESULTATS.setdefault("legendage", {})["CIDEr-D"] = tableau_lex.loc["BLIP (réelle)", "CIDEr-D"] if LEGENDE_BLIP else float("nan")
interpretation(f"La phrase fausse (« bus ») devance la paraphrase fidèle sur {len(devance)} métrique(s) sur {tableau_lex.shape[1]} "
               f"({', '.join(devance) or 'aucune'}) : elle reprend presque mot pour mot une référence. "
               "BLEU, METEOR et CIDEr mesurent un **recouvrement lexical**, pas l'exactitude de la scène.")

In [ ]:
def scorer_ma_legende(legende):
    if not tokeniser(legende):
        print("Écrivez une légende en anglais.")
        return
    s = tous_les_scores(legende)
    mots_refs = {w for r in REFS_EVAL for w in tokeniser(r)}
    display(HTML("<p style='font-size:16px'>" + " ".join(
        f"<b style='color:{PALETTE['sauge'] if w in mots_refs else PALETTE['terracotta']}'>{w}</b>" for w in tokeniser(legende)) + "</p>"))
    display(pd.Series(s, name="score").to_frame().T.style.format("{:.3f}", na_rep="—"))


if WIDGETS_OK:
    explorer(scorer_ma_legende, legende=widgets.Text(value="people wait for the subway on a platform", description="légende",
                                                    continuous_update=False, layout=widgets.Layout(width="620px")))
else:
    scorer_ma_legende("people wait for the subway on a platform")

### Conclusion
Les métriques **lexicales** (BLEU, METEOR, CIDEr) comptent des mots communs avec des légendes humaines : simples et reproductibles, mais aveugles au sens. Sur une seule image, les scores sont instables ; on les calcule sur des milliers d'images, avec plusieurs références chacune. **Langue** : BLIP écrit en anglais et n'est comparé qu'aux références anglaises.

---
# 5. Recherche image-texte — diapositives 16 à 18

**Objectif.** Classer les 12 photos de la collection de la diapositive 16 pour plusieurs requêtes, puis mesurer le classement avec Recall@K, Precision@K et MRR sur **4 requêtes étiquetées à la main**.

**Modèle** : CLIP (double encodeur). **Convention** : Recall@K et MRR utilisent l'unique *image associée* à chaque requête ; Precision@K utilise toutes les images *pertinentes* ; les images « partiellement pertinentes » comptent comme non pertinentes.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 3.6))
ax.set_xlim(0, 15)
ax.set_ylim(0, 3.6)
ax.set_axis_off()


def boite(x, y, l, h, texte, sous_texte="", plein=False):
    ax.add_patch(patches.FancyBboxPatch((x, y), l, h, boxstyle="round,pad=0.02,rounding_size=0.08",
                                        fc=PALETTE["sarcelle"] if plein else "white", ec=PALETTE["sarcelle"], lw=1.6))
    ax.text(x + l / 2, y + h / 2 + (0.15 if sous_texte else 0), texte, ha="center", va="center", fontsize=13.5,
            fontweight="bold", color="white" if plein else PALETTE["marine"])
    if sous_texte:
        ax.text(x + l / 2, y + h / 2 - 0.25, sous_texte, ha="center", va="center", fontsize=11,
                color="white" if plein else PALETTE["gris"])


def fleche(x1, y1, x2, y2):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="-|>", color=PALETTE["marine"], lw=1.6))


boite(0.1, 2.1, 2.2, 1.0, "Requête", "texte")
boite(0.1, 0.4, 2.2, 1.0, "Images", "collection")
boite(3.1, 2.1, 2.6, 1.0, "Encodeur texte", "Transformer")
boite(3.1, 0.4, 2.6, 1.0, "Encodeur image", "ViT-B/32")
boite(6.8, 0.9, 3.2, 1.7, "Espace partagé", "similarité cosinus", plein=True)
boite(11.0, 0.9, 2.2, 1.7, "Classement", "top K")
fleche(2.35, 2.6, 3.05, 2.6)
fleche(2.35, 0.9, 3.05, 0.9)
fleche(5.75, 2.6, 6.75, 2.1)
fleche(5.75, 0.9, 6.75, 1.4)
fleche(10.05, 1.75, 10.95, 1.75)
ax.text(13.4, 1.75, "images triées\npar similarité\ndécroissante", va="center", fontsize=11.5, color=PALETTE["gris"])
fig.suptitle("Recherche par double encodeur (type CLIP)", x=0.01, ha="left", fontsize=15, fontweight="bold")
etiqueter(fig, "conceptuel")
plt.show()

### 5.1 Entrée : la collection et les requêtes
Étiquettes attribuées par l'auteur à partir des légendes Flickr8k (ce n'est pas une vérité terrain officielle). Les requêtes françaises sont traduites en anglais pour CLIP ; la section 5.4 mesure l'effet de ce choix.

In [ ]:
collection = json.loads((ASSETS / "recherche" / "collection.json").read_text(encoding="utf-8"))
ids_images = [im["id"] for im in collection["images"]]
images_collection = [Image.open(ASSETS / "recherche" / im["fichier"]).convert("RGB") for im in collection["images"]]
requetes = collection["requetes"]
numero = {k: i + 1 for i, k in enumerate(ids_images)}  # numéro d'affichage de chaque image

fig, axes = plt.subplots(3, 4, figsize=(18, 13))
for ax, im, info in zip(axes.ravel(), images_collection, collection["images"]):
    ax.imshow(vignette(im))
    ax.set_axis_off()
    ax.set_title(f"n° {numero[info['id']]}", fontsize=14, fontweight="bold")
    ax.text(0.5, -0.04, textwrap.fill(info["description_fr"], 36), transform=ax.transAxes, ha="center", va="top", fontsize=11.5)
fig.suptitle("Collection de démonstration : 12 photos Flickr8k", x=0.01, ha="left", fontweight="bold")
fig.subplots_adjust(top=0.93, hspace=0.42, wspace=0.06)
etiqueter(fig, "donnees")
plt.show()

display(pd.DataFrame([{"requête": q["id"], "texte (diapositive)": q["texte_fr"], "texte envoyé à CLIP": q["texte_en"],
                       "image associée": f"n° {numero[q['image_associee']]}",
                       "pertinentes": ", ".join(f"n° {numero[k]}" for k in q["pertinentes"]),
                       "partiellement pertinentes": ", ".join(f"n° {numero[k]}" for k in q["partiellement_pertinentes"]) or "—"}
                      for q in requetes]))

### 5.2 Prédiction : similarités et classements réels
Cadre **sarcelle épais** : image associée · **sauge** : pertinente · **ocre** : partiellement pertinente · **gris** : non pertinente.

In [ ]:
if "CLIP" not in globals():
    with chrono("Chargement de CLIP"):
        CLIP = charger_clip()
print(f"Modèle : {CLIP['nom']}")

with chrono("Encodage des 12 images et des 4 requêtes"):
    vecteurs_collection = encoder_images(CLIP, images_collection)
    vecteurs_requetes = encoder_textes(CLIP, [q["texte_en"] for q in requetes])
similarites = (vecteurs_requetes @ vecteurs_collection.T).numpy()  # matrice 4 × 12 de cosinus


def classer(scores):
    # Renvoie les identifiants d'images triés par similarité décroissante
    return [ids_images[i] for i in np.argsort(-scores, kind="stable")]


classements = {q["id"]: classer(s) for q, s in zip(requetes, similarites)}
rangs = {q["id"]: classements[q["id"]].index(q["image_associee"]) + 1 for q in requetes}
print("Rang de l'image associée :", ", ".join(f"{k} → {r}" for k, r in rangs.items()))

In [ ]:
def niveau(q, image_id):
    if image_id == q["image_associee"]:
        return "associée", PALETTE["sarcelle"], 6
    if image_id in q["pertinentes"]:
        return "pertinente", PALETTE["sauge"], 4
    if image_id in q["partiellement_pertinentes"]:
        return "partielle", PALETTE["ocre"], 4
    return "non pertinente", PALETTE["gris"], 2


K_AFFICHE = 5
fig, axes = plt.subplots(len(requetes), K_AFFICHE + 1, figsize=(20, 3.5 * len(requetes)),
                         gridspec_kw={"width_ratios": [1.5] + [1] * K_AFFICHE})
for ligne, (q, scores) in enumerate(zip(requetes, similarites)):
    axes[ligne, 0].set_axis_off()
    axes[ligne, 0].text(0, 0.62, f"{q['id']}", fontsize=16, fontweight="bold", transform=axes[ligne, 0].transAxes)
    axes[ligne, 0].text(0, 0.5, textwrap.fill(q["texte_fr"], 24), fontsize=12.5, transform=axes[ligne, 0].transAxes, va="top")
    axes[ligne, 0].text(0, 0.12, f"rang de l'image associée : {rangs[q['id']]}", fontsize=12, color=PALETTE["sarcelle"],
                        fontweight="bold", transform=axes[ligne, 0].transAxes)
    for k, image_id in enumerate(classements[q["id"]][:K_AFFICHE]):
        ax = axes[ligne, k + 1]
        ax.imshow(vignette(images_collection[numero[image_id] - 1]))
        ax.set_xticks([])
        ax.set_yticks([])
        nom, couleur, epaisseur = niveau(q, image_id)
        for cote in ax.spines.values():
            cote.set_visible(True)
            cote.set_edgecolor(couleur)
            cote.set_linewidth(epaisseur)
        ax.set_title(f"rang {k + 1} · n° {numero[image_id]} · cos = {fmt(scores[numero[image_id] - 1], 3)}", fontsize=11)
        ax.set_xlabel(nom, fontsize=11.5, color=couleur, fontweight="bold")
fig.suptitle(f"Classement réel des images pour chaque requête ({CLIP['nom']})", x=0.01, ha="left", fontweight="bold")
etiqueter(fig, "reel", CLIP["nom"])
fig.subplots_adjust(top=0.93, hspace=0.42, wspace=0.08)
sauvegarder(fig, "recherche_classements.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(16, 3.8))
im = ax.imshow(similarites, cmap="BuGn", aspect="auto")
for i, q in enumerate(requetes):
    for j, image_id in enumerate(ids_images):
        ax.text(j, i, fmt(similarites[i, j], 2), ha="center", va="center", fontsize=10.5,
                color="white" if similarites[i, j] > np.percentile(similarites, 80) else PALETTE["marine"])
    j = ids_images.index(q["image_associee"])
    ax.add_patch(patches.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec=PALETTE["terracotta"], lw=3))
ax.set_xticks(range(len(ids_images)), [f"n° {k}" for k in range(1, len(ids_images) + 1)])
ax.set_yticks(range(len(requetes)), [q["id"] for q in requetes])
ax.set_xlabel("Image de la collection")
ax.set_title("Similarité cosinus requête × image (cadre terracotta : image associée)", loc="left")
plt.colorbar(im, ax=ax, label="cos θ")
etiqueter(fig, "reel", CLIP["nom"])
plt.show()

### 5.3 Évaluation

$$\text{Recall@}K = \frac{1}{|Q|}\sum_{q}\mathbb{1}[\text{rang}_q \le K] \qquad \text{Precision@}K = \frac{|\text{pertinentes} \cap \text{top }K|}{K} \qquad \text{MRR} = \frac{1}{|Q|}\sum_q \frac{1}{\text{rang}_q}$$

**Plus haut = meilleur** pour les trois. Exemple de la diapositive 18 (rangs construits à la main) :

In [ ]:
def recall_at_k(liste_rangs, k):
    return float(np.mean([r <= k for r in liste_rangs]))


def mrr(liste_rangs):
    return float(np.mean([1 / r for r in liste_rangs]))


rangs_diapo = [2, 1, 3, 7]
print(f"Rangs : {rangs_diapo}")
for k in [1, 3, 5]:
    print(f"Recall@{k} = {sum(r <= k for r in rangs_diapo)}/4 = {fmt(recall_at_k(rangs_diapo, k), pct=True)}")
print(f"MRR = (1/2 + 1/1 + 1/3 + 1/7) / 4 = {fmt(mrr(rangs_diapo), 3)}")

**Mesure réelle** (rangs obtenus par CLIP, non ajustés pour retrouver les 75 % de la diapositive ; 4 requêtes et 12 images : une démonstration, pas un benchmark).

In [ ]:
rangs_mesures = list(rangs.values())
lignes = []
for k in [1, 3, 5]:
    precisions = [len(set(classements[q["id"]][:k]) & set(q["pertinentes"])) / k for q in requetes]
    rappels = [len(set(classements[q["id"]][:k]) & set(q["pertinentes"])) / len(q["pertinentes"]) for q in requetes]
    lignes.append({"K": k, "Recall@K (image associée)": recall_at_k(rangs_mesures, k),
                   "Precision@K (pertinentes)": np.mean(precisions), "Rappel général@K": np.mean(rappels)})
tableau_recherche = pd.DataFrame(lignes).set_index("K")
display(tableau_recherche.style.format("{:.3f}"))
print(f"Rangs réels de l'image associée : {rangs} · MRR = {fmt(mrr(rangs_mesures), 3)}")
RESULTATS["recherche"] = {f"R@{k}": recall_at_k(rangs_mesures, k) for k in [1, 3, 5]}
RESULTATS["recherche"].update({"MRR": mrr(rangs_mesures), "modele": CLIP["nom"]})

fig, axes = plt.subplots(1, 2, figsize=(17, 4.8))
ks = np.arange(1, len(ids_images) + 1)
axes[0].step(ks, [recall_at_k(rangs_mesures, k) for k in ks], where="post", color=PALETTE["sarcelle"], lw=2.6, label="mesuré (CLIP)")
axes[0].step(ks, [recall_at_k(rangs_diapo, k) for k in ks], where="post", color=PALETTE["ocre"], lw=2, ls="--",
             label="exemple de la diapositive (pédagogique)")
axes[0].set_xticks(ks)
axes[0].set_ylim(0, 1.05)
axes[0].set_xlabel("K (nombre de résultats examinés)")
axes[0].set_ylabel("Recall@K")
axes[0].set_title("Recall@K croît avec K : on compare toujours au même K", loc="left", fontsize=13)
axes[0].legend(loc="lower right")
x = np.arange(3)
axes[1].bar(x - 0.2, tableau_recherche["Recall@K (image associée)"], width=0.4, color=PALETTE["sarcelle"], label="Recall@K (image associée)")
axes[1].bar(x + 0.2, tableau_recherche["Precision@K (pertinentes)"], width=0.4, color=PALETTE["sauge"], label="Precision@K (pertinentes)")
for xi, (a, b) in enumerate(zip(tableau_recherche.iloc[:, 0], tableau_recherche.iloc[:, 1])):
    axes[1].text(xi - 0.2, a + 0.02, fmt(a), ha="center", fontsize=12)
    axes[1].text(xi + 0.2, b + 0.02, fmt(b), ha="center", fontsize=12)
axes[1].set_xticks(x, [f"K = {k}" for k in [1, 3, 5]])
axes[1].set_ylim(0, 1.15)
axes[1].legend(loc="upper left")
axes[1].set_title("Deux questions différentes : « trouvée ? » et « quelle part du top K est pertinente ? »", loc="left", fontsize=13)
etiqueter(fig, "reel", f"{CLIP['nom']} · 12 images, 4 requêtes")
sauvegarder(fig, "recherche_metriques.png")
plt.show()

q_echec = [q for q in requetes if rangs[q["id"]] > 3]
texte = (f"Recall@3 mesuré = {fmt(RESULTATS['recherche']['R@3'], pct=True)}, MRR = {fmt(RESULTATS['recherche']['MRR'], 3)}. ")
for q in q_echec:
    top3 = classements[q["id"]][:3]
    n_pert = len(set(top3) & set(q["pertinentes"]))
    texte += (f"Pour {q['id']}, l'image associée n'arrive qu'au rang {rangs[q['id']]} ; pourtant {n_pert} des 3 premières images sont "
              "pertinentes : la convention « une seule image associée » compte comme un échec un classement qu'un humain jugerait utile. ")
interpretation(texte + "Toute coïncidence avec les 75 % de la diapositive serait fortuite : ces valeurs dépendent du modèle et de la collection.")

### 5.4 Effet de la langue de la requête

In [ ]:
vecteurs_fr = encoder_textes(CLIP, [q["texte_fr"] for q in requetes])
similarites_fr = (vecteurs_fr @ vecteurs_collection.T).numpy()
rangs_fr = {q["id"]: classer(s).index(q["image_associee"]) + 1 for q, s in zip(requetes, similarites_fr)}
comparaison_langue = pd.DataFrame({
    "rang (anglais)": rangs, "rang (français)": rangs_fr,
    "cos image associée (anglais)": {q["id"]: similarites[i, ids_images.index(q["image_associee"])] for i, q in enumerate(requetes)},
    "cos image associée (français)": {q["id"]: similarites_fr[i, ids_images.index(q["image_associee"])] for i, q in enumerate(requetes)},
})
display(comparaison_langue.style.format({"cos image associée (anglais)": "{:.3f}", "cos image associée (français)": "{:.3f}"}))
print(f"Recall@3 : anglais {fmt(recall_at_k(list(rangs.values()), 3), pct=True)} · français {fmt(recall_at_k(list(rangs_fr.values()), 3), pct=True)}")
interpretation("Le choix de la langue modifie les similarités et parfois les rangs. Pour une application en français, on utiliserait "
               "un modèle multilingue ; dans tous les cas, la langue des requêtes fait partie du protocole à documenter.")

### 5.5 À vous : votre requête, votre K

In [ ]:
TEXTES_REQUETES = {f"{q['id']} · {q['texte_fr']}": q for q in requetes}


def chercher(requete, texte_libre, k):
    q = TEXTES_REQUETES.get(requete)
    texte = q["texte_en"] if q else texte_libre
    if not texte.strip():
        print("Écrivez une requête (en anglais de préférence).")
        return
    scores = (encoder_textes(CLIP, [texte]) @ vecteurs_collection.T).numpy().ravel()
    ordre = np.argsort(-scores, kind="stable")[:k]
    fig, axes = plt.subplots(1, k, figsize=(max(3.2 * k, 11), 3.6), squeeze=False)
    for rang, (ax, i) in enumerate(zip(axes[0], ordre), start=1):
        ax.imshow(vignette(images_collection[i]))
        ax.set_xticks([])
        ax.set_yticks([])
        if q:
            nom, couleur, epaisseur = niveau(q, ids_images[i])
            for cote in ax.spines.values():
                cote.set_visible(True)
                cote.set_edgecolor(couleur)
                cote.set_linewidth(epaisseur)
            ax.set_xlabel(nom, color=couleur, fontweight="bold")
        ax.set_title(f"rang {rang} · cos = {fmt(scores[i], 3)}", fontsize=11)
    titre = f"« {texte} » — top {k}"
    sous_titre = ""
    if q:
        rang_assoc = list(np.argsort(-scores, kind="stable")).index(ids_images.index(q["image_associee"])) + 1
        pert = len({ids_images[i] for i in ordre} & set(q["pertinentes"]))
        sous_titre = f"\nimage associée au rang {rang_assoc} · Recall@{k} = {int(rang_assoc <= k)} · Precision@{k} = {fmt(pert / k)}"
    fig.suptitle(titre + sous_titre, x=0.01, ha="left", fontsize=12.5, fontweight="bold")
    fig.subplots_adjust(top=0.7)
    etiqueter(fig, "reel", CLIP["nom"])


if WIDGETS_OK:
    explorer(chercher,
             requete=widgets.Dropdown(options=list(TEXTES_REQUETES) + ["requête libre"], description="requête"),
             texte_libre=widgets.Text(value="a dog in the rain", description="texte libre", continuous_update=False),
             k=widgets.IntSlider(value=3, min=1, max=6, description="K"))
else:
    chercher(list(TEXTES_REQUETES)[0], "", 3)

### Conclusion
Recall@K répond à « l'image attendue est-elle dans les K premiers ? » et ignore le rang exact : le **MRR** le complète. Precision@K répond à « quelle part du top K est pertinente ? ». Les valeurs dépendent de K, de la taille de la collection, de la langue… et de la **convention de pertinence**, qui doit toujours être annoncée.

---
# Synthèse — Comparaison des cinq tâches et de leurs métriques (diapositive 19)

Tableau construit **uniquement** à partir des résultats calculés dans cette exécution ; « non calculé » signifie que le modèle n'a pas pu être exécuté ici (matériel, réseau).

In [ ]:
def cellule(cle_tache, fonction, defaut="non calculé dans cette exécution"):
    r = RESULTATS.get(cle_tache)
    try:
        return fonction(r) if r else defaut
    except (KeyError, TypeError):
        return defaut


g = RESULTATS.get("generation", {})
lignes_synthese = [
    ["Suivi d'objets", "YOLO11n + ByteTrack", "vidéo de trafic", "boîtes + identifiants", "MOTA ↑ (IDF1 ↑, HOTA ↑)",
     cellule("suivi", lambda r: f"MOTA = {fmt(r['MOTA'], 3)} ; IDF1 = {fmt(r['IDF1'], 3)} ; HOTA = {fmt(r.get('HOTA'), 3)} (MOT17-04, 8 images)"),
     "erreurs (FN, FP, IDSW) rapportées aux objets réels", "dominé par la détection ; identités peu pénalisées"],
    ["Génération d'images", "Stable Diffusion" if g.get("image_generee") else "Stable Diffusion (non exécuté) ; CLIP",
     "prompt", "image", "FID ↓ ; CLIPScore ↑",
     "FID : non calculé (il faut des milliers d'images) ; " + cellule("generation", lambda r: (
         f"CLIPScore de l'image générée = {fmt(r['clipscore_image_generee'], 3)}" if r.get("image_generee") else
         f"cos(photo réelle, prompt) = {fmt(r['cosinus_photo'], 3)}") + f" ({r['modele_clip']})"),
     "FID : écart entre distributions ; CLIPScore : accord texte-image", "FID ignore le texte ; CLIPScore hérite des biais de CLIP"],
    ["OCR", "Tesseract 5 (LSTM)", "facture numérisée", "texte", "CER ↓ (WER ↓, EM ↑)",
     cellule("ocr", lambda r: f"CER = {fmt(r['CER'], pct=True)} ; WER = {fmt(r['WER'], pct=True)} (scan de la présentation, "
                              f"normalisation stricte) ; CER dégradé = {fmt(r['CER_degrade'], pct=True)}"),
     "modifications par caractère, rapportées à la référence", "tous les caractères pèsent pareil ; dépend de la normalisation"],
    ["Légendage", "BLIP base", "photo", "phrase (anglais)", "BLEU ↑ ; METEOR ↑ ; CIDEr ↑",
     cellule("legendage", lambda r: f"BLEU-1 = {fmt(r['BLEU-1'], 3)} ; METEOR = {fmt(r['METEOR'], 3)} (5 références) ; "
                                    f"CIDEr-D = {fmt(r.get('CIDEr-D'), 2)} (4 références)") if RESULTATS.get("legendage", {}).get("legende") else "BLIP non exécuté : non calculé",
     "recouvrement de mots avec des légendes humaines", "lexical : ignore le sens et les synonymes"],
    ["Recherche image-texte", cellule("recherche", lambda r: r["modele"], "CLIP"), "requête écrite", "images classées",
     "Recall@K ↑ ; Precision@K ↑ ; MRR ↑",
     cellule("recherche", lambda r: f"R@1 = {fmt(r['R@1'])} ; R@3 = {fmt(r['R@3'])} ; R@5 = {fmt(r['R@5'])} ; "
                                    f"MRR = {fmt(r['MRR'], 3)} (12 images, 4 requêtes)"),
     "part des requêtes dont l'image associée est dans le top K", "ignore le rang exact ; pertinence binaire et conventionnelle"],
]
entetes = ["Tâche", "Modèle ou approche", "Entrée", "Sortie", "Métrique principale", "Résultat mesuré (contexte)",
           "Ce que mesure le score", "Limite principale"]
tableau_md = "| " + " | ".join(entetes) + " |\n|" + "---|" * len(entetes) + "\n"
tableau_md += "\n".join("| " + " | ".join(str(c) for c in ligne) + " |" for ligne in lignes_synthese)
display(Markdown(tableau_md))

Ces scores **ne se comparent pas entre eux** : échelles et sens différents (MOTA peut être négatif, le CER peut dépasser 100 %, le FID n'a pas de maximum), références différentes (vérité exacte, plusieurs réponses acceptables, distributions) et protocoles différents (seuil d'IoU, normalisation, K, langue, nombre d'images). Une métrique se compare uniquement à elle-même, sur la même tâche, les mêmes données et le même protocole.

> **Une métrique n'est utile que si l'on comprend ce qu'elle mesure et ses limites.**
>
> **Une bonne évaluation ne consiste pas uniquement à obtenir un score élevé, mais à comprendre ce que mesure ce score et dans quelles conditions il reste pertinent.**

---
## Annexe — Reproductibilité
Versions utilisées dans cette exécution et figures enregistrées dans `resultats/`.

In [ ]:
import importlib.metadata as metadonnees

versions = {"Python": platform.python_version(), "Matériel": DEVICE.upper(), "Mode": MODE}
for paquet in ["torch", "torchvision", "ultralytics", "open_clip_torch", "transformers", "diffusers", "motmetrics", "jiwer",
               "nltk", "pytesseract", "scikit-learn", "ipywidgets", "opencv-python", "opencv-python-headless", "numpy", "pandas", "scipy", "matplotlib"]:
    try:
        versions[paquet] = metadonnees.version(paquet)
    except metadonnees.PackageNotFoundError:
        versions[paquet] = "absent"
versions["Tesseract (programme)"] = globals().get("VERSION_TESSERACT", "non vérifié") if globals().get("TESSERACT_OK") else "absent ou non vérifié"
display(pd.Series(versions, name="version").to_frame())
print("Figures enregistrées :", ", ".join(sorted(p.name for p in SORTIES.glob("*.png"))) or "aucune")

## Références principales

- Bernardin, K. & Stiefelhagen, R. (2008). *Evaluating multiple object tracking performance: the CLEAR MOT metrics.* EURASIP JIVP.
- Ristani, E. et al. (2016). *Performance measures and a data set for multi-target, multi-camera tracking* (IDF1). ECCV Workshops.
- Luiten, J. et al. (2021). *HOTA: A higher order metric for evaluating multi-object tracking.* IJCV 129.
- Zhang, Y. et al. (2022). *ByteTrack: Multi-object tracking by associating every detection box.* ECCV.
- Milan, A. et al. (2016). *MOT16: A benchmark for multi-object tracking.* arXiv:1603.00831.
- Caesar, H. et al. (2020). *nuScenes: A multimodal dataset for autonomous driving.* CVPR.
- Ho, J., Jain, A. & Abbeel, P. (2020). *Denoising diffusion probabilistic models.* NeurIPS.
- Rombach, R. et al. (2022). *High-resolution image synthesis with latent diffusion models.* CVPR.
- Heusel, M. et al. (2017). *GANs trained by a two time-scale update rule converge to a local Nash equilibrium* (FID). NeurIPS.
- Salimans, T. et al. (2016). *Improved techniques for training GANs* (Inception Score). NeurIPS.
- Radford, A. et al. (2021). *Learning transferable visual models from natural language supervision* (CLIP). ICML.
- Hessel, J. et al. (2021). *CLIPScore: A reference-free evaluation metric for image captioning.* EMNLP.
- Jocher, G. & Qiu, J. (2024). *Ultralytics YOLO11* (logiciel, AGPL-3.0).
- Smith, R. (2007). *An overview of the Tesseract OCR engine.* ICDAR.
- Levenshtein, V. I. (1966). *Binary codes capable of correcting deletions, insertions, and reversals.* Soviet Physics Doklady 10(8).
- Li, J. et al. (2022). *BLIP: Bootstrapping language-image pre-training for unified vision-language understanding and generation.* ICML.
- Papineni, K. et al. (2002). *BLEU: a method for automatic evaluation of machine translation.* ACL.
- Banerjee, S. & Lavie, A. (2005). *METEOR: An automatic metric for MT evaluation with improved correlation with human judgments.* ACL Workshop.
- Vedantam, R., Zitnick, C. L. & Parikh, D. (2015). *CIDEr: Consensus-based image description evaluation.* CVPR.
- Hodosh, M., Young, P. & Hockenmaier, J. (2013). *Framing image description as a ranking task* (Flickr8k). JAIR 47.
- Voorhees, E. M. (1999). *The TREC-8 question answering track report* (MRR). TREC-8.

Provenance et licences des données : voir `assets/SOURCES.md`.